In [1]:
from __future__ import annotations

import json
import re
import subprocess
import sys
from pathlib import Path
from typing import Dict, Any, List, Optional

import pandas as pd


# ============================================================
# CHECK AND INSTALL MISSING DEPENDENCIES
# ============================================================

def install_package(package: str):
    """Install a package using pip with user flag."""
    print(f"Installing {package}...")
    try:
        # Try --user flag first (installs only for current user)
        subprocess.check_call([sys.executable, "-m", "pip", "install", "--user", package])
        print(f"{package} installed successfully.")
    except subprocess.CalledProcessError:
        # If --user fails, try with --break-system-packages (last resort)
        print(f"Trying with --break-system-packages for {package}...")
        try:
            subprocess.check_call([sys.executable, "-m", "pip", "install", "--break-system-packages", package])
            print(f"{package} installed successfully.")
        except subprocess.CalledProcessError:
            print(f"Failed to install {package}. Please install manually with:")
            print(f"  pip install --user {package}")
            print(f"Or create a virtual environment.")
            sys.exit(1)


try:
    import xlsxwriter
except ImportError:
    print("xlsxwriter not found. Installing...")
    install_package("xlsxwriter")
    import xlsxwriter


# ============================================================
# CONFIG
# ============================================================

ROOT = Path.cwd()

RUN_DIR = ROOT / "runs_transformer_fractional"
OUT_XLSX = ROOT / "transformer_fractional_results.xlsx"

print("Current folder:", ROOT)
print("Run dir:", RUN_DIR)
print("Output Excel:", OUT_XLSX)


# ============================================================
# HELPERS
# ============================================================

def read_json(path: Path) -> Optional[Dict[str, Any]]:
    if not path.exists():
        return None

    try:
        with open(path, "r", encoding="utf-8") as f:
            return json.load(f)
    except Exception as e:
        print(f"Failed to read JSON: {path} | {e}")
        return None


def read_text(path: Path) -> str:
    if not path.exists():
        return ""

    try:
        return path.read_text(encoding="utf-8", errors="ignore")
    except Exception:
        return ""


def flatten_dict(d: Dict[str, Any], prefix: str = "") -> Dict[str, Any]:
    out = {}

    if not isinstance(d, dict):
        return out

    for k, v in d.items():
        key = f"{prefix}{k}" if prefix == "" else f"{prefix}_{k}"

        if isinstance(v, dict):
            out.update(flatten_dict(v, key))
        elif isinstance(v, (list, tuple)):
            out[key] = json.dumps(v, ensure_ascii=False)
        else:
            out[key] = v

    return out


def parse_best_from_log(log_text: str) -> Dict[str, Any]:
    out = {}

    patterns = {
        "log_best_epoch": r"BEST EPOCH\s*:\s*([0-9]+)",
        "log_best_val_macro_f1": r"BEST VAL MACRO F1\s*:\s*([0-9eE\.\+\-]+)",
        "log_best_val_acc": r"BEST VAL ACC\s*:\s*([0-9eE\.\+\-]+)",
        "log_best_val_loss": r"BEST VAL LOSS\s*:\s*([0-9eE\.\+\-]+)",
        "log_total_run_time_sec": r"TOTAL RUN TIME\s*:\s*([0-9eE\.\+\-]+)",
    }

    for name, pat in patterns.items():
        m = re.search(pat, log_text)
        if m:
            val = m.group(1)

            try:
                if "epoch" in name:
                    out[name] = int(val)
                else:
                    out[name] = float(val)
            except Exception:
                out[name] = val

    return out


def infer_from_path(run_path: Path) -> Dict[str, Any]:
    """
    Expected structure:
        runs_transformer_fractional / dataset / model / optimizer / run_0
    """

    rel = run_path.relative_to(RUN_DIR)
    parts = rel.parts

    out = {
        "run_folder": str(run_path),
        "relative_run_folder": str(rel),
    }

    if len(parts) >= 4:
        out["path_dataset_name"] = parts[0]
        out["path_model_name"] = parts[1]
        out["path_optimizer_name"] = parts[2]
        out["path_run_id"] = parts[3]

        m = re.search(r"run_(\d+)", parts[3])
        out["run_id_num"] = int(m.group(1)) if m else None

    return out


def get_last(history: Dict[str, Any], key: str):
    arr = history.get(key, None)

    if isinstance(arr, list) and len(arr) > 0:
        return arr[-1]

    return None


def get_best_epoch_idx(history: Dict[str, Any], metric: str = "val_macro_f1") -> Optional[int]:
    arr = history.get(metric, None)

    if not isinstance(arr, list) or len(arr) == 0:
        return None

    numeric = pd.to_numeric(pd.Series(arr), errors="coerce")

    if numeric.notna().sum() == 0:
        return None

    return int(numeric.idxmax())


def autosize_worksheet(writer, sheet_name: str, df: pd.DataFrame, max_width: int = 42):
    worksheet = writer.sheets[sheet_name]

    if df.empty:
        return

    for idx, col in enumerate(df.columns):
        series = df[col].astype(str)
        sample = series.head(2000).tolist()
        max_len = max([len(str(col))] + [len(x) for x in sample])
        width = min(max(max_len + 2, 10), max_width)
        worksheet.set_column(idx, idx, width)


def safe_to_numeric(df: pd.DataFrame, cols: List[str]) -> pd.DataFrame:
    for c in cols:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")
    return df


# ============================================================
# COLLECT RUN FOLDERS
# ============================================================

if not RUN_DIR.exists():
    raise FileNotFoundError(f"Run directory not found: {RUN_DIR}")

run_folders = sorted([
    p for p in RUN_DIR.rglob("run_*")
    if p.is_dir()
])

print("Found run folders:", len(run_folders))

if len(run_folders) == 0:
    raise RuntimeError("No run_* folders found.")


# ============================================================
# PARSE ALL RUNS
# ============================================================

summary_rows: List[Dict[str, Any]] = []
epoch_rows: List[Dict[str, Any]] = []
file_rows: List[Dict[str, Any]] = []

for run_path in run_folders:
    dataset_info_path = run_path / "dataset_info.json"
    model_config_path = run_path / "model_config.json"
    optimizer_config_path = run_path / "optimizer_config.json"
    history_path = run_path / "history.json"
    log_path = run_path / "log.txt"

    initial_model_path = run_path / "initial_model.pt"
    best_model_path = run_path / "best_model.pt"
    final_model_path = run_path / "final_model.pt"

    dataset_info = read_json(dataset_info_path) or {}
    model_config = read_json(model_config_path) or {}
    optimizer_config = read_json(optimizer_config_path) or {}
    history = read_json(history_path) or {}
    log_text = read_text(log_path)

    path_info = infer_from_path(run_path)
    log_best = parse_best_from_log(log_text)

    best_idx = get_best_epoch_idx(history, metric="val_macro_f1")

    row = {}
    row.update(path_info)

    # Main configs
    row.update(flatten_dict(dataset_info, "dataset"))
    row.update(flatten_dict(model_config, "model"))
    row.update(flatten_dict(optimizer_config, "optimizer"))

    # File existence
    row["has_dataset_info_json"] = dataset_info_path.exists()
    row["has_model_config_json"] = model_config_path.exists()
    row["has_optimizer_config_json"] = optimizer_config_path.exists()
    row["has_history_json"] = history_path.exists()
    row["has_log_txt"] = log_path.exists()
    row["has_initial_model_pt"] = initial_model_path.exists()
    row["has_best_model_pt"] = best_model_path.exists()
    row["has_final_model_pt"] = final_model_path.exists()

    row["history_path"] = str(history_path)
    row["log_path"] = str(log_path)
    row["initial_model_path"] = str(initial_model_path)
    row["best_model_path"] = str(best_model_path)
    row["final_model_path"] = str(final_model_path)

    # Epoch count
    if isinstance(history, dict) and "val_macro_f1" in history and isinstance(history["val_macro_f1"], list):
        row["n_epochs_recorded"] = len(history["val_macro_f1"])
    else:
        row["n_epochs_recorded"] = 0

    # Final metrics
    metric_keys = [
        "train_loss",
        "train_accuracy",
        "train_macro_precision",
        "train_macro_recall",
        "train_macro_f1",
        "train_micro_f1",
        "train_weighted_f1",
        "train_confidence",

        "val_loss",
        "val_accuracy",
        "val_macro_precision",
        "val_macro_recall",
        "val_macro_f1",
        "val_micro_f1",
        "val_weighted_f1",
        "val_confidence",

        "grad_norm",
        "epoch_time",
        "throughput_samples_per_sec",
        "lr",
    ]

    for key in metric_keys:
        row[f"final_{key}"] = get_last(history, key)

    # Best metrics by val_macro_f1
    if best_idx is not None:
        row["best_epoch_by_history"] = best_idx + 1

        for key in metric_keys:
            values = history.get(key, None)
            if isinstance(values, list) and best_idx < len(values):
                row[f"best_{key}"] = values[best_idx]
            else:
                row[f"best_{key}"] = None
    else:
        row["best_epoch_by_history"] = None

        for key in metric_keys:
            row[f"best_{key}"] = None

    # Values parsed from log
    row.update(log_best)

    # Runtime aggregates
    if isinstance(history.get("epoch_time"), list) and len(history["epoch_time"]) > 0:
        row["mean_epoch_time"] = sum(history["epoch_time"]) / len(history["epoch_time"])
        row["total_epoch_time"] = sum(history["epoch_time"])
    else:
        row["mean_epoch_time"] = None
        row["total_epoch_time"] = None

    if isinstance(history.get("throughput_samples_per_sec"), list) and len(history["throughput_samples_per_sec"]) > 0:
        row["mean_throughput_samples_per_sec"] = (
            sum(history["throughput_samples_per_sec"])
            / len(history["throughput_samples_per_sec"])
        )
    else:
        row["mean_throughput_samples_per_sec"] = None

    # Completion flags
    row["is_complete_10_epochs"] = row["n_epochs_recorded"] >= 10
    row["has_valid_best_val_macro_f1"] = row.get("best_val_macro_f1") is not None

    summary_rows.append(row)

    # Epoch-level table
    if isinstance(history, dict) and len(history) > 0:
        max_epochs = max(
            [len(v) for v in history.values() if isinstance(v, list)] or [0]
        )

        for epoch_idx in range(max_epochs):
            erow = {}
            erow.update(path_info)

            erow["epoch"] = epoch_idx + 1

            erow["dataset_name"] = dataset_info.get(
                "dataset_name",
                path_info.get("path_dataset_name"),
            )
            erow["model_name"] = model_config.get(
                "model_name",
                path_info.get("path_model_name"),
            )
            erow["optimizer_name"] = optimizer_config.get(
                "optimizer_name",
                path_info.get("path_optimizer_name"),
            )

            erow["d_model"] = model_config.get("d_model")
            erow["nhead"] = model_config.get("nhead")
            erow["num_layers"] = model_config.get("num_layers")
            erow["dim_feedforward"] = model_config.get("dim_feedforward")
            erow["dropout"] = model_config.get("dropout")
            erow["pooling"] = model_config.get("pooling")

            erow["fractional"] = optimizer_config.get("fractional")
            erow["fractional_alpha"] = optimizer_config.get("fractional_alpha")
            erow["fractional_beta"] = optimizer_config.get("fractional_beta")
            erow["lr_config"] = optimizer_config.get("lr")

            for key, values in history.items():
                if isinstance(values, list):
                    erow[key] = values[epoch_idx] if epoch_idx < len(values) else None

            epoch_rows.append(erow)

    # File index
    for f in [
        dataset_info_path,
        model_config_path,
        optimizer_config_path,
        history_path,
        log_path,
        initial_model_path,
        best_model_path,
        final_model_path,
    ]:
        file_rows.append(
            {
                "run_folder": str(run_path),
                "file_name": f.name,
                "file_path": str(f),
                "exists": f.exists(),
                "size_bytes": f.stat().st_size if f.exists() else None,
            }
        )


summary_df = pd.DataFrame(summary_rows)
epochs_df = pd.DataFrame(epoch_rows)
files_df = pd.DataFrame(file_rows)

print("summary_df:", summary_df.shape)
print("epochs_df :", epochs_df.shape)
print("files_df  :", files_df.shape)


# ============================================================
# NUMERIC CONVERSION
# ============================================================

numeric_cols = [
    c for c in summary_df.columns
    if any(
        x in c.lower()
        for x in [
            "f1",
            "accuracy",
            "loss",
            "confidence",
            "time",
            "throughput",
            "lr",
            "dropout",
            "weight_decay",
            "alpha",
            "beta",
            "epoch",
            "samples",
            "seq_len",
            "vocab_size",
            "num_classes",
            "d_model",
            "nhead",
            "num_layers",
            "params",
        ]
    )
]

summary_df = safe_to_numeric(summary_df, numeric_cols)

if not epochs_df.empty:
    epoch_numeric_cols = [
        c for c in epochs_df.columns
        if c not in [
            "run_folder",
            "relative_run_folder",
            "path_dataset_name",
            "path_model_name",
            "path_optimizer_name",
            "path_run_id",
            "dataset_name",
            "model_name",
            "optimizer_name",
            "pooling",
        ]
    ]
    epochs_df = safe_to_numeric(epochs_df, epoch_numeric_cols)


# ============================================================
# CLEAN / SORT SUMMARY
# ============================================================

preferred_summary_cols = [
    "path_dataset_name",
    "path_model_name",
    "path_optimizer_name",
    "path_run_id",
    "run_id_num",

    "dataset_dataset_name",
    "dataset_n_samples",
    "dataset_seq_len",
    "dataset_vocab_size",
    "dataset_num_classes",
    "dataset_mean_length",
    "dataset_max_length",

    "model_model_name",
    "model_d_model",
    "model_nhead",
    "model_num_layers",
    "model_dim_feedforward",
    "model_dropout",
    "model_pooling",

    "optimizer_optimizer_name",
    "optimizer_base_optimizer",
    "optimizer_lr",
    "optimizer_weight_decay",
    "optimizer_fractional",
    "optimizer_fractional_alpha",
    "optimizer_fractional_beta",

    "n_epochs_recorded",
    "is_complete_10_epochs",
    "has_valid_best_val_macro_f1",

    "best_epoch_by_history",
    "best_val_macro_f1",
    "best_val_accuracy",
    "best_val_loss",
    "best_val_weighted_f1",
    "best_val_micro_f1",
    "best_val_confidence",

    "final_val_macro_f1",
    "final_val_accuracy",
    "final_val_loss",
    "final_val_weighted_f1",
    "final_val_micro_f1",
    "final_val_confidence",

    "final_train_macro_f1",
    "final_train_accuracy",
    "final_train_loss",

    "mean_epoch_time",
    "total_epoch_time",
    "mean_throughput_samples_per_sec",

    "has_initial_model_pt",
    "has_best_model_pt",
    "has_final_model_pt",

    "history_path",
    "log_path",
    "best_model_path",
    "final_model_path",
]

summary_cols = [c for c in preferred_summary_cols if c in summary_df.columns]
summary_cols += [c for c in summary_df.columns if c not in summary_cols]
summary_df = summary_df[summary_cols]

sort_cols = [
    c for c in [
        "path_dataset_name",
        "path_model_name",
        "path_optimizer_name",
        "run_id_num",
        "path_run_id",
    ]
    if c in summary_df.columns
]

if sort_cols:
    summary_df = summary_df.sort_values(sort_cols).reset_index(drop=True)

if not epochs_df.empty:
    sort_cols_epochs = [
        c for c in [
            "dataset_name",
            "model_name",
            "optimizer_name",
            "run_id_num",
            "path_run_id",
            "epoch",
        ]
        if c in epochs_df.columns
    ]

    if sort_cols_epochs:
        epochs_df = epochs_df.sort_values(sort_cols_epochs).reset_index(drop=True)


# ============================================================
# BEST BY GROUP — SAFE VERSION
# ============================================================

group_cols = [
    c for c in [
        "path_dataset_name",
        "path_model_name",
        "path_optimizer_name",
    ]
    if c in summary_df.columns
]

if group_cols and "best_val_macro_f1" in summary_df.columns:
    tmp = summary_df.copy()

    tmp["best_val_macro_f1_num"] = pd.to_numeric(
        tmp["best_val_macro_f1"],
        errors="coerce",
    )

    valid_tmp = tmp.dropna(subset=["best_val_macro_f1_num"]).copy()

    if len(valid_tmp) > 0:
        idx = valid_tmp.groupby(group_cols)["best_val_macro_f1_num"].idxmax()

        best_by_group_df = (
            valid_tmp
            .loc[idx]
            .drop(columns=["best_val_macro_f1_num"])
            .reset_index(drop=True)
        )

        best_by_group_df = best_by_group_df.sort_values(
            "best_val_macro_f1",
            ascending=False,
        ).reset_index(drop=True)
    else:
        best_by_group_df = pd.DataFrame()

    valid_groups = set(
        tuple(x)
        for x in valid_tmp[group_cols].drop_duplicates().to_numpy()
    )

    all_groups = set(
        tuple(x)
        for x in tmp[group_cols].drop_duplicates().to_numpy()
    )

    bad_groups = sorted(all_groups - valid_groups)

    incomplete_groups_df = pd.DataFrame(
        bad_groups,
        columns=group_cols,
    )

else:
    best_by_group_df = pd.DataFrame()
    incomplete_groups_df = pd.DataFrame()


# ============================================================
# INCOMPLETE RUNS
# ============================================================

incomplete_runs_df = summary_df.copy()

if "best_val_macro_f1" in incomplete_runs_df.columns:
    incomplete_runs_df["best_val_macro_f1_num"] = pd.to_numeric(
        incomplete_runs_df["best_val_macro_f1"],
        errors="coerce",
    )

    incomplete_runs_df = incomplete_runs_df[
        (incomplete_runs_df["n_epochs_recorded"].fillna(0) < 10)
        | (incomplete_runs_df["best_val_macro_f1_num"].isna())
        | (~incomplete_runs_df["has_history_json"].astype(bool))
    ].copy()

    incomplete_runs_df = incomplete_runs_df.drop(
        columns=["best_val_macro_f1_num"],
        errors="ignore",
    )
else:
    incomplete_runs_df = pd.DataFrame()


# ============================================================
# PIVOT / AGGREGATED MEANS
# ============================================================

agg_cols = [
    "best_val_macro_f1",
    "best_val_accuracy",
    "best_val_loss",
    "final_val_macro_f1",
    "final_val_accuracy",
    "final_val_loss",
    "mean_epoch_time",
    "mean_throughput_samples_per_sec",
]

available_agg_cols = [c for c in agg_cols if c in summary_df.columns]

if group_cols and available_agg_cols:
    pivot_source = summary_df.copy()

    for c in available_agg_cols:
        pivot_source[c] = pd.to_numeric(pivot_source[c], errors="coerce")

    pivot_mean_df = (
        pivot_source
        .groupby(group_cols, dropna=False)[available_agg_cols]
        .agg(["count", "mean", "std", "min", "max"])
        .reset_index()
    )

    pivot_mean_df.columns = [
        "_".join([str(x) for x in col if str(x) != ""])
        for col in pivot_mean_df.columns
    ]

    if "best_val_macro_f1_mean" in pivot_mean_df.columns:
        pivot_mean_df = pivot_mean_df.sort_values(
            "best_val_macro_f1_mean",
            ascending=False,
        ).reset_index(drop=True)
else:
    pivot_mean_df = pd.DataFrame()


# ============================================================
# OVERALL RANKING
# ============================================================

if "best_val_macro_f1" in summary_df.columns:
    ranking_df = summary_df.copy()

    ranking_df["best_val_macro_f1_num"] = pd.to_numeric(
        ranking_df["best_val_macro_f1"],
        errors="coerce",
    )

    ranking_df = (
        ranking_df
        .dropna(subset=["best_val_macro_f1_num"])
        .sort_values("best_val_macro_f1_num", ascending=False)
        .drop(columns=["best_val_macro_f1_num"])
        .reset_index(drop=True)
    )
else:
    ranking_df = pd.DataFrame()


# ============================================================
# PROGRESS OVERVIEW
# ============================================================

expected_without_lion = 3 * 5 * 8 * 3
expected_with_lion = 3 * 5 * 9 * 3

progress_rows = [
    {
        "metric": "found_run_folders",
        "value": len(run_folders),
    },
    {
        "metric": "summary_rows",
        "value": len(summary_df),
    },
    {
        "metric": "epoch_rows",
        "value": len(epochs_df),
    },
    {
        "metric": "incomplete_run_rows",
        "value": len(incomplete_runs_df),
    },
    {
        "metric": "incomplete_group_rows",
        "value": len(incomplete_groups_df),
    },
    {
        "metric": "expected_experiments_without_lion",
        "value": expected_without_lion,
    },
    {
        "metric": "expected_experiments_with_lion",
        "value": expected_with_lion,
    },
    {
        "metric": "completion_vs_360_percent",
        "value": 100.0 * len(run_folders) / expected_without_lion,
    },
    {
        "metric": "completion_vs_405_percent",
        "value": 100.0 * len(run_folders) / expected_with_lion,
    },
]

progress_df = pd.DataFrame(progress_rows)


# ============================================================
# WRITE EXCEL — FALLBACK TO OPENPYXL IF XLSXWRITER FAILS
# ============================================================

try:
    # Try with xlsxwriter first
    with pd.ExcelWriter(OUT_XLSX, engine="xlsxwriter") as writer:
        summary_df.to_excel(writer, sheet_name="summary_all", index=False)
        epochs_df.to_excel(writer, sheet_name="epochs_all", index=False)
        best_by_group_df.to_excel(writer, sheet_name="best_by_group", index=False)
        pivot_mean_df.to_excel(writer, sheet_name="pivot_mean", index=False)
        ranking_df.to_excel(writer, sheet_name="ranking_all", index=False)
        incomplete_runs_df.to_excel(writer, sheet_name="incomplete_runs", index=False)
        incomplete_groups_df.to_excel(writer, sheet_name="incomplete_groups", index=False)
        files_df.to_excel(writer, sheet_name="files_index", index=False)
        progress_df.to_excel(writer, sheet_name="progress", index=False)

        workbook = writer.book

        header_format = workbook.add_format(
            {
                "bold": True,
                "bg_color": "#D9EAF7",
                "border": 1,
                "text_wrap": True,
                "valign": "top",
            }
        )

        number_format = workbook.add_format({"num_format": "0.0000"})
        int_format = workbook.add_format({"num_format": "0"})

        sheets = [
            ("summary_all", summary_df),
            ("epochs_all", epochs_df),
            ("best_by_group", best_by_group_df),
            ("pivot_mean", pivot_mean_df),
            ("ranking_all", ranking_df),
            ("incomplete_runs", incomplete_runs_df),
            ("incomplete_groups", incomplete_groups_df),
            ("files_index", files_df),
            ("progress", progress_df),
        ]

        for sheet_name, df in sheets:
            worksheet = writer.sheets[sheet_name]

            if df.empty:
                continue

            # Header style
            for col_idx, col_name in enumerate(df.columns):
                worksheet.write(0, col_idx, col_name, header_format)

            worksheet.freeze_panes(1, 0)
            worksheet.autofilter(0, 0, max(len(df), 1), max(len(df.columns) - 1, 0))

            autosize_worksheet(writer, sheet_name, df)

            # Numeric formatting
            for col_idx, col_name in enumerate(df.columns):
                cname = str(col_name).lower()

                if any(
                    x in cname
                    for x in [
                        "f1",
                        "accuracy",
                        "loss",
                        "confidence",
                        "time",
                        "throughput",
                        "lr",
                        "dropout",
                        "weight_decay",
                        "alpha",
                        "beta",
                        "mean",
                        "std",
                        "min",
                        "max",
                    ]
                ):
                    worksheet.set_column(col_idx, col_idx, None, number_format)

                if any(
                    x in cname
                    for x in [
                        "epoch",
                        "samples",
                        "seq_len",
                        "vocab_size",
                        "num_classes",
                        "d_model",
                        "nhead",
                        "num_layers",
                        "params",
                        "count",
                        "rows",
                        "folders",
                    ]
                ):
                    worksheet.set_column(col_idx, col_idx, None, int_format)

                if any(x in cname for x in ["path", "folder"]):
                    worksheet.set_column(col_idx, col_idx, 42)

        # Conditional formatting
        def apply_scale(sheet_name: str, df: pd.DataFrame, col_name: str):
            if df.empty or col_name not in df.columns:
                return

            col = df.columns.get_loc(col_name)
            writer.sheets[sheet_name].conditional_format(
                1,
                col,
                len(df),
                col,
                {
                    "type": "3_color_scale",
                },
            )

        apply_scale("summary_all", summary_df, "best_val_macro_f1")
        apply_scale("summary_all", summary_df, "final_val_macro_f1")
        apply_scale("best_by_group", best_by_group_df, "best_val_macro_f1")
        apply_scale("ranking_all", ranking_df, "best_val_macro_f1")

        if not pivot_mean_df.empty and "best_val_macro_f1_mean" in pivot_mean_df.columns:
            apply_scale("pivot_mean", pivot_mean_df, "best_val_macro_f1_mean")

    print("\nDONE")
    print("Excel saved to:", OUT_XLSX)

except Exception as e:
    print(f"\nError with xlsxwriter: {e}")
    print("Trying with openpyxl engine instead...")
    
    # Fallback to openpyxl
    try:
        with pd.ExcelWriter(OUT_XLSX, engine="openpyxl") as writer:
            summary_df.to_excel(writer, sheet_name="summary_all", index=False)
            epochs_df.to_excel(writer, sheet_name="epochs_all", index=False)
            best_by_group_df.to_excel(writer, sheet_name="best_by_group", index=False)
            pivot_mean_df.to_excel(writer, sheet_name="pivot_mean", index=False)
            ranking_df.to_excel(writer, sheet_name="ranking_all", index=False)
            incomplete_runs_df.to_excel(writer, sheet_name="incomplete_runs", index=False)
            incomplete_groups_df.to_excel(writer, sheet_name="incomplete_groups", index=False)
            files_df.to_excel(writer, sheet_name="files_index", index=False)
            progress_df.to_excel(writer, sheet_name="progress", index=False)
        
        print("\nDONE with openpyxl")
        print("Excel saved to:", OUT_XLSX)
        
    except ImportError:
        print("\nopenpyxl also not found. Installing openpyxl...")
        install_package("openpyxl")
        
        with pd.ExcelWriter(OUT_XLSX, engine="openpyxl") as writer:
            summary_df.to_excel(writer, sheet_name="summary_all", index=False)
            epochs_df.to_excel(writer, sheet_name="epochs_all", index=False)
            best_by_group_df.to_excel(writer, sheet_name="best_by_group", index=False)
            pivot_mean_df.to_excel(writer, sheet_name="pivot_mean", index=False)
            ranking_df.to_excel(writer, sheet_name="ranking_all", index=False)
            incomplete_runs_df.to_excel(writer, sheet_name="incomplete_runs", index=False)
            incomplete_groups_df.to_excel(writer, sheet_name="incomplete_groups", index=False)
            files_df.to_excel(writer, sheet_name="files_index", index=False)
            progress_df.to_excel(writer, sheet_name="progress", index=False)
        
        print("\nDONE with openpyxl")
        print("Excel saved to:", OUT_XLSX)


print("\nDiagnostics:")
print("Found run folders:", len(run_folders))
print("Summary rows:", len(summary_df))
print("Epoch rows:", len(epochs_df))
print("Incomplete runs:", len(incomplete_runs_df))
print("Incomplete groups:", len(incomplete_groups_df))

print("\nExpected:")
print("Without Lion:", expected_without_lion)
print("With Lion   :", expected_with_lion)

print("\nCompletion:")
print(f"vs 360: {100.0 * len(run_folders) / expected_without_lion:.2f}%")
print(f"vs 405: {100.0 * len(run_folders) / expected_with_lion:.2f}%")

Current folder: /home/tahiti/Malashin_Projects
Run dir: /home/tahiti/Malashin_Projects/runs_transformer_fractional
Output Excel: /home/tahiti/Malashin_Projects/transformer_fractional_results.xlsx
Found run folders: 360
summary_df: (360, 105)
epochs_df : (3600, 41)
files_df  : (2880, 5)

Error with xlsxwriter: object of type 'float' has no len()
Trying with openpyxl engine instead...

DONE with openpyxl
Excel saved to: /home/tahiti/Malashin_Projects/transformer_fractional_results.xlsx

Diagnostics:
Found run folders: 360
Summary rows: 360
Epoch rows: 3600
Incomplete runs: 0
Incomplete groups: 0

Expected:
Without Lion: 360
With Lion   : 405

Completion:
vs 360: 100.00%
vs 405: 88.89%


In [2]:
from pathlib import Path
import shutil

# Папка, где сейчас работает Jupyter
cwd = Path.cwd()

# Информация о диске/разделе
total, used, free = shutil.disk_usage(cwd)

def gb(x):
    return x / (1024 ** 3)

print(f"Текущая папка Jupyter: {cwd}")
print(f"Диск / раздел: {cwd.anchor}")
print(f"Всего:      {gb(total):.2f} GB")
print(f"Использовано: {gb(used):.2f} GB")
print(f"Свободно:  {gb(free):.2f} GB")

Текущая папка Jupyter: /home/tahiti/Malashin_Projects
Диск / раздел: /
Всего:      1006.85 GB
Использовано: 171.07 GB
Свободно:  784.57 GB


In [3]:
from __future__ import annotations

import json
import re
from pathlib import Path
from typing import Dict, Any, List, Optional

import pandas as pd


# ============================================================
# CONFIG
# ============================================================

ROOT = Path.cwd()
RUN_DIR = ROOT / "runs_transformer_fractional"
OUT_XLSX = ROOT / "experiment_statistics.xlsx"
OUT_CSV = ROOT / "experiment_statistics_summary.csv"

EXPECTED_EPOCHS = 10
RUNS = 3

print("ROOT:", ROOT)
print("RUN_DIR:", RUN_DIR)
print("OUT_XLSX:", OUT_XLSX)


# ============================================================
# SAME CONFIGS AS TRAINING SCRIPT
# ============================================================

DATASET_CONFIGS = [
    {
        "dataset_name": "small_D2_L64",
        "n_samples": 5000,
        "depth": 2,
        "max_len": 64,
        "min_args": 2,
        "max_args": 4,
    },
    {
        "dataset_name": "base_D3_L128",
        "n_samples": 10000,
        "depth": 3,
        "max_len": 128,
        "min_args": 2,
        "max_args": 4,
    },
    {
        "dataset_name": "hard_D4_L160",
        "n_samples": 12000,
        "depth": 4,
        "max_len": 160,
        "min_args": 2,
        "max_args": 4,
    },
]

MODEL_CONFIGS = [
    {
        "model_name": "T_small_mean",
        "d_model": 96,
        "nhead": 4,
        "num_layers": 2,
        "dim_feedforward": 256,
        "dropout": 0.10,
        "pooling": "mean",
    },
    {
        "model_name": "T_base_mean",
        "d_model": 128,
        "nhead": 4,
        "num_layers": 3,
        "dim_feedforward": 512,
        "dropout": 0.10,
        "pooling": "mean",
    },
    {
        "model_name": "T_base_cls",
        "d_model": 128,
        "nhead": 4,
        "num_layers": 3,
        "dim_feedforward": 512,
        "dropout": 0.10,
        "pooling": "cls",
    },
    {
        "model_name": "T_deep_mean",
        "d_model": 128,
        "nhead": 4,
        "num_layers": 5,
        "dim_feedforward": 512,
        "dropout": 0.15,
        "pooling": "mean",
    },
    {
        "model_name": "T_wide_mean",
        "d_model": 192,
        "nhead": 6,
        "num_layers": 3,
        "dim_feedforward": 768,
        "dropout": 0.10,
        "pooling": "mean",
    },
]

OPTIMIZER_CONFIGS = [
    {
        "optimizer_name": "adam",
        "base_optimizer": "adam",
        "lr": 3e-4,
        "weight_decay": 0.0,
        "fractional": False,
        "fractional_alpha": None,
        "fractional_beta": None,
    },
    {
        "optimizer_name": "adamw",
        "base_optimizer": "adamw",
        "lr": 3e-4,
        "weight_decay": 1e-2,
        "fractional": False,
        "fractional_alpha": None,
        "fractional_beta": None,
    },
    {
        "optimizer_name": "sgd",
        "base_optimizer": "sgd",
        "lr": 1e-2,
        "momentum": 0.9,
        "weight_decay": 0.0,
        "fractional": False,
        "fractional_alpha": None,
        "fractional_beta": None,
    },
    {
        "optimizer_name": "rmsprop",
        "base_optimizer": "rmsprop",
        "lr": 1e-3,
        "weight_decay": 0.0,
        "fractional": False,
        "fractional_alpha": None,
        "fractional_beta": None,
    },
    {
        "optimizer_name": "adagrad",
        "base_optimizer": "adagrad",
        "lr": 1e-2,
        "weight_decay": 0.0,
        "fractional": False,
        "fractional_alpha": None,
        "fractional_beta": None,
    },
    {
        "optimizer_name": "adamw_frac_a07",
        "base_optimizer": "adamw",
        "lr": 3e-4,
        "weight_decay": 1e-2,
        "fractional": True,
        "fractional_alpha": 0.70,
        "fractional_beta": 0.90,
    },
    {
        "optimizer_name": "adamw_frac_a08",
        "base_optimizer": "adamw",
        "lr": 3e-4,
        "weight_decay": 1e-2,
        "fractional": True,
        "fractional_alpha": 0.80,
        "fractional_beta": 0.90,
    },
    {
        "optimizer_name": "adamw_frac_a09",
        "base_optimizer": "adamw",
        "lr": 3e-4,
        "weight_decay": 1e-2,
        "fractional": True,
        "fractional_alpha": 0.90,
        "fractional_beta": 0.90,
    },
]


# ============================================================
# HELPERS
# ============================================================

def safe_name(x: Any) -> str:
    return (
        str(x)
        .replace(" ", "_")
        .replace("/", "_")
        .replace("\\", "_")
        .replace(":", "_")
        .replace(".", "p")
        .replace("'", "")
        .replace('"', "")
    )


def read_json(path: Path) -> Optional[Dict[str, Any]]:
    if not path.exists():
        return None

    try:
        with open(path, "r", encoding="utf-8") as f:
            return json.load(f)
    except Exception:
        return None


def get_exp_dir(dataset_name: str, model_name: str, optimizer_name: str, run_id: int) -> Path:
    return (
        RUN_DIR
        / safe_name(dataset_name)
        / safe_name(model_name)
        / safe_name(optimizer_name)
        / f"run_{run_id}"
    )


def is_complete(exp_dir: Path) -> bool:
    history_path = exp_dir / "history.json"
    best_model_path = exp_dir / "best_model.pt"
    final_model_path = exp_dir / "final_model.pt"

    if not history_path.exists():
        return False
    if not best_model_path.exists():
        return False
    if not final_model_path.exists():
        return False

    history = read_json(history_path)
    if not history:
        return False

    vals = history.get("val_macro_f1", None)
    if not isinstance(vals, list):
        return False

    return len(vals) >= EXPECTED_EPOCHS


def get_history_metrics(exp_dir: Path) -> Dict[str, Any]:
    history_path = exp_dir / "history.json"
    history = read_json(history_path)

    if not history:
        return {
            "n_epochs_recorded": 0,
            "best_epoch": None,
            "best_val_macro_f1": None,
            "best_val_accuracy": None,
            "best_val_loss": None,
            "final_val_macro_f1": None,
            "final_val_accuracy": None,
            "final_val_loss": None,
            "final_train_macro_f1": None,
            "final_train_accuracy": None,
            "mean_epoch_time": None,
            "total_epoch_time": None,
            "mean_throughput": None,
        }

    val_macro_f1 = history.get("val_macro_f1", [])
    n_epochs = len(val_macro_f1) if isinstance(val_macro_f1, list) else 0

    if n_epochs == 0:
        best_idx = None
    else:
        s = pd.to_numeric(pd.Series(val_macro_f1), errors="coerce")
        best_idx = int(s.idxmax()) if s.notna().sum() > 0 else None

    def at_best(key):
        values = history.get(key, [])
        if best_idx is not None and isinstance(values, list) and best_idx < len(values):
            return values[best_idx]
        return None

    def last(key):
        values = history.get(key, [])
        if isinstance(values, list) and len(values) > 0:
            return values[-1]
        return None

    epoch_times = history.get("epoch_time", [])
    throughputs = history.get("throughput_samples_per_sec", [])

    return {
        "n_epochs_recorded": n_epochs,

        "best_epoch": best_idx + 1 if best_idx is not None else None,
        "best_val_macro_f1": at_best("val_macro_f1"),
        "best_val_accuracy": at_best("val_accuracy"),
        "best_val_loss": at_best("val_loss"),
        "best_val_weighted_f1": at_best("val_weighted_f1"),
        "best_val_micro_f1": at_best("val_micro_f1"),
        "best_val_confidence": at_best("val_confidence"),

        "final_val_macro_f1": last("val_macro_f1"),
        "final_val_accuracy": last("val_accuracy"),
        "final_val_loss": last("val_loss"),
        "final_val_weighted_f1": last("val_weighted_f1"),
        "final_val_micro_f1": last("val_micro_f1"),
        "final_val_confidence": last("val_confidence"),

        "final_train_macro_f1": last("train_macro_f1"),
        "final_train_accuracy": last("train_accuracy"),
        "final_train_loss": last("train_loss"),

        "mean_epoch_time": (
            sum(epoch_times) / len(epoch_times)
            if isinstance(epoch_times, list) and len(epoch_times) > 0
            else None
        ),
        "total_epoch_time": (
            sum(epoch_times)
            if isinstance(epoch_times, list) and len(epoch_times) > 0
            else None
        ),
        "mean_throughput": (
            sum(throughputs) / len(throughputs)
            if isinstance(throughputs, list) and len(throughputs) > 0
            else None
        ),
    }


# ============================================================
# BUILD EXPECTED JOB TABLE
# ============================================================

rows = []

for dcfg in DATASET_CONFIGS:
    for mcfg in MODEL_CONFIGS:
        for ocfg in OPTIMIZER_CONFIGS:
            for run_id in range(RUNS):
                dataset_name = dcfg["dataset_name"]
                model_name = mcfg["model_name"]
                optimizer_name = ocfg["optimizer_name"]

                exp_dir = get_exp_dir(
                    dataset_name=dataset_name,
                    model_name=model_name,
                    optimizer_name=optimizer_name,
                    run_id=run_id,
                )

                history_path = exp_dir / "history.json"
                best_model_path = exp_dir / "best_model.pt"
                final_model_path = exp_dir / "final_model.pt"
                initial_model_path = exp_dir / "initial_model.pt"

                metrics = get_history_metrics(exp_dir)

                complete = is_complete(exp_dir)

                if complete:
                    status = "complete"
                elif exp_dir.exists():
                    status = "incomplete_or_broken"
                else:
                    status = "missing"

                row = {
                    "status": status,
                    "is_complete": complete,

                    "dataset_name": dataset_name,
                    "n_samples": dcfg["n_samples"],
                    "depth": dcfg["depth"],
                    "max_len": dcfg["max_len"],
                    "min_args": dcfg["min_args"],
                    "max_args": dcfg["max_args"],

                    "model_name": model_name,
                    "d_model": mcfg["d_model"],
                    "nhead": mcfg["nhead"],
                    "num_layers": mcfg["num_layers"],
                    "dim_feedforward": mcfg["dim_feedforward"],
                    "dropout": mcfg["dropout"],
                    "pooling": mcfg["pooling"],

                    "optimizer_name": optimizer_name,
                    "base_optimizer": ocfg.get("base_optimizer"),
                    "lr": ocfg.get("lr"),
                    "weight_decay": ocfg.get("weight_decay"),
                    "fractional": ocfg.get("fractional"),
                    "fractional_alpha": ocfg.get("fractional_alpha"),
                    "fractional_beta": ocfg.get("fractional_beta"),

                    "run_id": run_id,

                    "exp_dir": str(exp_dir),
                    "has_exp_dir": exp_dir.exists(),
                    "has_history_json": history_path.exists(),
                    "has_initial_model_pt": initial_model_path.exists(),
                    "has_best_model_pt": best_model_path.exists(),
                    "has_final_model_pt": final_model_path.exists(),
                }

                row.update(metrics)
                rows.append(row)

stats_df = pd.DataFrame(rows)

print("Expected jobs:", len(stats_df))
print(stats_df["status"].value_counts(dropna=False))


# ============================================================
# BASIC OVERVIEW
# ============================================================

total_expected = len(stats_df)
n_complete = int((stats_df["status"] == "complete").sum())
n_incomplete = int((stats_df["status"] == "incomplete_or_broken").sum())
n_missing = int((stats_df["status"] == "missing").sum())

overview_df = pd.DataFrame([
    {"metric": "expected_total_jobs", "value": total_expected},
    {"metric": "complete_jobs", "value": n_complete},
    {"metric": "incomplete_or_broken_jobs", "value": n_incomplete},
    {"metric": "missing_jobs", "value": n_missing},
    {"metric": "completion_percent", "value": 100.0 * n_complete / total_expected},
    {"metric": "found_any_folder_percent", "value": 100.0 * (n_complete + n_incomplete) / total_expected},
    {"metric": "expected_epochs_total", "value": total_expected * EXPECTED_EPOCHS},
    {"metric": "recorded_epochs_total", "value": int(stats_df["n_epochs_recorded"].fillna(0).sum())},
])


# ============================================================
# SUBTABLES
# ============================================================

complete_df = stats_df[stats_df["status"] == "complete"].copy()
incomplete_df = stats_df[stats_df["status"] == "incomplete_or_broken"].copy()
missing_df = stats_df[stats_df["status"] == "missing"].copy()

leaderboard_df = (
    complete_df
    .sort_values("best_val_macro_f1", ascending=False)
    .reset_index(drop=True)
)

best_by_combo_df = (
    complete_df
    .sort_values("best_val_macro_f1", ascending=False)
    .groupby(["dataset_name", "model_name", "optimizer_name"], as_index=False)
    .head(1)
    .reset_index(drop=True)
)

agg_cols = [
    "best_val_macro_f1",
    "best_val_accuracy",
    "best_val_loss",
    "final_val_macro_f1",
    "final_val_accuracy",
    "final_val_loss",
    "mean_epoch_time",
    "mean_throughput",
]

aggregate_df = (
    complete_df
    .groupby(
        [
            "dataset_name",
            "model_name",
            "optimizer_name",
            "fractional",
            "fractional_alpha",
        ],
        dropna=False,
    )[agg_cols]
    .agg(["count", "mean", "std", "min", "max"])
    .reset_index()
)

aggregate_df.columns = [
    "_".join([str(x) for x in col if str(x) != ""])
    for col in aggregate_df.columns
]

if "best_val_macro_f1_mean" in aggregate_df.columns:
    aggregate_df = aggregate_df.sort_values(
        "best_val_macro_f1_mean",
        ascending=False,
    ).reset_index(drop=True)


by_optimizer_df = (
    complete_df
    .groupby(["optimizer_name", "fractional", "fractional_alpha"], dropna=False)[
        [
            "best_val_macro_f1",
            "best_val_accuracy",
            "best_val_loss",
            "mean_epoch_time",
        ]
    ]
    .agg(["count", "mean", "std", "min", "max"])
    .reset_index()
)

by_optimizer_df.columns = [
    "_".join([str(x) for x in col if str(x) != ""])
    for col in by_optimizer_df.columns
]

if "best_val_macro_f1_mean" in by_optimizer_df.columns:
    by_optimizer_df = by_optimizer_df.sort_values(
        "best_val_macro_f1_mean",
        ascending=False,
    ).reset_index(drop=True)


by_model_df = (
    complete_df
    .groupby(["model_name", "d_model", "num_layers", "pooling"], dropna=False)[
        [
            "best_val_macro_f1",
            "best_val_accuracy",
            "best_val_loss",
            "mean_epoch_time",
        ]
    ]
    .agg(["count", "mean", "std", "min", "max"])
    .reset_index()
)

by_model_df.columns = [
    "_".join([str(x) for x in col if str(x) != ""])
    for col in by_model_df.columns
]

if "best_val_macro_f1_mean" in by_model_df.columns:
    by_model_df = by_model_df.sort_values(
        "best_val_macro_f1_mean",
        ascending=False,
    ).reset_index(drop=True)


by_dataset_df = (
    complete_df
    .groupby(["dataset_name", "n_samples", "depth", "max_len"], dropna=False)[
        [
            "best_val_macro_f1",
            "best_val_accuracy",
            "best_val_loss",
            "mean_epoch_time",
        ]
    ]
    .agg(["count", "mean", "std", "min", "max"])
    .reset_index()
)

by_dataset_df.columns = [
    "_".join([str(x) for x in col if str(x) != ""])
    for col in by_dataset_df.columns
]


# ============================================================
# SAVE
# ============================================================

stats_df.to_csv(OUT_CSV, index=False)

with pd.ExcelWriter(OUT_XLSX, engine="openpyxl") as writer:
    overview_df.to_excel(writer, sheet_name="overview", index=False)
    stats_df.to_excel(writer, sheet_name="all_expected_jobs", index=False)
    complete_df.to_excel(writer, sheet_name="complete_runs", index=False)
    incomplete_df.to_excel(writer, sheet_name="incomplete_runs", index=False)
    missing_df.to_excel(writer, sheet_name="missing_runs", index=False)
    leaderboard_df.to_excel(writer, sheet_name="leaderboard", index=False)
    best_by_combo_df.to_excel(writer, sheet_name="best_by_combo", index=False)
    aggregate_df.to_excel(writer, sheet_name="aggregate_combo", index=False)
    by_optimizer_df.to_excel(writer, sheet_name="by_optimizer", index=False)
    by_model_df.to_excel(writer, sheet_name="by_model", index=False)
    by_dataset_df.to_excel(writer, sheet_name="by_dataset", index=False)

print("\nDONE")
print("CSV:", OUT_CSV)
print("Excel:", OUT_XLSX)

print("\nOverview:")
print(overview_df)

print("\nTop 10 leaderboard:")
display_cols = [
    "dataset_name",
    "model_name",
    "optimizer_name",
    "run_id",
    "best_val_macro_f1",
    "best_val_accuracy",
    "best_val_loss",
    "best_epoch",
]
print(leaderboard_df[display_cols].head(10))

ROOT: /home/tahiti/Malashin_Projects
RUN_DIR: /home/tahiti/Malashin_Projects/runs_transformer_fractional
OUT_XLSX: /home/tahiti/Malashin_Projects/experiment_statistics.xlsx
Expected jobs: 360
status
complete    360
Name: count, dtype: int64

DONE
CSV: /home/tahiti/Malashin_Projects/experiment_statistics_summary.csv
Excel: /home/tahiti/Malashin_Projects/experiment_statistics.xlsx

Overview:
                      metric   value
0        expected_total_jobs   360.0
1              complete_jobs   360.0
2  incomplete_or_broken_jobs     0.0
3               missing_jobs     0.0
4         completion_percent   100.0
5   found_any_folder_percent   100.0
6      expected_epochs_total  3600.0
7      recorded_epochs_total  3600.0

Top 10 leaderboard:
   dataset_name   model_name optimizer_name  run_id  best_val_macro_f1  \
0  small_D2_L64  T_wide_mean          adamw       0           0.322226   
1  small_D2_L64  T_wide_mean          adamw       2           0.313547   
2  small_D2_L64  T_wide_mean   

In [2]:
from __future__ import annotations

import json
import re
from pathlib import Path
from typing import Dict, Any, Optional, List

import pandas as pd


# ============================================================
# CONFIG
# ============================================================

ROOT = Path.cwd()

RUN_DIR = ROOT / "runs_fractional_scenarios"

OUT_XLSX = ROOT / "fractional_scenarios_statistics.xlsx"
OUT_CSV = ROOT / "fractional_scenarios_all_jobs.csv"

EXPECTED_EPOCHS = 10
RUNS = 3

print("ROOT:", ROOT)
print("RUN_DIR:", RUN_DIR)
print("OUT_XLSX:", OUT_XLSX)


# ============================================================
# SAME CONFIGS AS EXPERIMENT
# ============================================================

DATASET_CONFIGS = [
    {
        "dataset_name": "small_D2_L64",
        "n_samples": 5000,
        "depth": 2,
        "max_len": 64,
        "seed_offset": 0,
    },
    {
        "dataset_name": "base_D3_L128",
        "n_samples": 10000,
        "depth": 3,
        "max_len": 128,
        "seed_offset": 1000,
    },
    {
        "dataset_name": "hard_D4_L160",
        "n_samples": 12000,
        "depth": 4,
        "max_len": 160,
        "seed_offset": 2000,
    },
]

MODEL_CONFIGS = [
    {
        "model_name": "T_small_mean",
        "d_model": 96,
        "nhead": 4,
        "num_layers": 2,
        "dim_feedforward": 256,
        "dropout": 0.10,
        "pooling": "mean",
    },
    {
        "model_name": "T_base_mean",
        "d_model": 128,
        "nhead": 4,
        "num_layers": 3,
        "dim_feedforward": 512,
        "dropout": 0.10,
        "pooling": "mean",
    },
    {
        "model_name": "T_base_cls",
        "d_model": 128,
        "nhead": 4,
        "num_layers": 3,
        "dim_feedforward": 512,
        "dropout": 0.10,
        "pooling": "cls",
    },
    {
        "model_name": "T_deep_mean",
        "d_model": 128,
        "nhead": 4,
        "num_layers": 5,
        "dim_feedforward": 512,
        "dropout": 0.15,
        "pooling": "mean",
    },
    {
        "model_name": "T_wide_mean",
        "d_model": 192,
        "nhead": 6,
        "num_layers": 3,
        "dim_feedforward": 768,
        "dropout": 0.10,
        "pooling": "mean",
    },
]

SCENARIO_CONFIGS = [
    {
        "scenario_name": "baseline_adamw",
        "base_optimizer": "adamw",
        "lr": 3e-4,
        "weight_decay": 1e-2,
        "mode": "none",
        "target": "none",
        "alpha": None,
        "beta": None,
        "mix_lambda": 0.0,
        "start_epoch": None,
    },
    {
        "scenario_name": "naive_all_replace_a08",
        "base_optimizer": "adamw",
        "lr": 3e-4,
        "weight_decay": 1e-2,
        "mode": "replace",
        "target": "all",
        "alpha": 0.80,
        "beta": 0.90,
        "mix_lambda": 1.0,
        "start_epoch": 1,
    },
    {
        "scenario_name": "delayed_all_replace_a08_warm3",
        "base_optimizer": "adamw",
        "lr": 3e-4,
        "weight_decay": 1e-2,
        "mode": "replace",
        "target": "all",
        "alpha": 0.80,
        "beta": 0.90,
        "mix_lambda": 1.0,
        "start_epoch": 4,
    },
    {
        "scenario_name": "mixed_all_a08_lam025",
        "base_optimizer": "adamw",
        "lr": 3e-4,
        "weight_decay": 1e-2,
        "mode": "mix",
        "target": "all",
        "alpha": 0.80,
        "beta": 0.90,
        "mix_lambda": 0.25,
        "start_epoch": 1,
    },
    {
        "scenario_name": "mixed_all_a08_lam050",
        "base_optimizer": "adamw",
        "lr": 3e-4,
        "weight_decay": 1e-2,
        "mode": "mix",
        "target": "all",
        "alpha": 0.80,
        "beta": 0.90,
        "mix_lambda": 0.50,
        "start_epoch": 1,
    },
    {
        "scenario_name": "head_only_replace_a08",
        "base_optimizer": "adamw",
        "lr": 3e-4,
        "weight_decay": 1e-2,
        "mode": "replace",
        "target": "head",
        "alpha": 0.80,
        "beta": 0.90,
        "mix_lambda": 1.0,
        "start_epoch": 1,
    },
    {
        "scenario_name": "ffn_only_replace_a08",
        "base_optimizer": "adamw",
        "lr": 3e-4,
        "weight_decay": 1e-2,
        "mode": "replace",
        "target": "ffn",
        "alpha": 0.80,
        "beta": 0.90,
        "mix_lambda": 1.0,
        "start_epoch": 1,
    },
    {
        "scenario_name": "attention_only_replace_a08",
        "base_optimizer": "adamw",
        "lr": 3e-4,
        "weight_decay": 1e-2,
        "mode": "replace",
        "target": "attention",
        "alpha": 0.80,
        "beta": 0.90,
        "mix_lambda": 1.0,
        "start_epoch": 1,
    },
    {
        "scenario_name": "embeddings_only_replace_a08",
        "base_optimizer": "adamw",
        "lr": 3e-4,
        "weight_decay": 1e-2,
        "mode": "replace",
        "target": "embeddings",
        "alpha": 0.80,
        "beta": 0.90,
        "mix_lambda": 1.0,
        "start_epoch": 1,
    },
    {
        "scenario_name": "delayed_mixed_all_a08_lam025_warm3",
        "base_optimizer": "adamw",
        "lr": 3e-4,
        "weight_decay": 1e-2,
        "mode": "mix",
        "target": "all",
        "alpha": 0.80,
        "beta": 0.90,
        "mix_lambda": 0.25,
        "start_epoch": 4,
    },
    {
        "scenario_name": "delayed_mixed_head_a08_lam025_warm3",
        "base_optimizer": "adamw",
        "lr": 3e-4,
        "weight_decay": 1e-2,
        "mode": "mix",
        "target": "head",
        "alpha": 0.80,
        "beta": 0.90,
        "mix_lambda": 0.25,
        "start_epoch": 4,
    },
]


# ============================================================
# HELPERS
# ============================================================

def safe_name(x: Any) -> str:
    return (
        str(x)
        .replace(" ", "_")
        .replace("/", "_")
        .replace("\\", "_")
        .replace(":", "_")
        .replace(".", "p")
        .replace("'", "")
        .replace('"', "")
    )


def read_json(path: Path) -> Optional[Dict[str, Any]]:
    if not path.exists():
        return None

    try:
        with open(path, "r", encoding="utf-8") as f:
            return json.load(f)
    except Exception as e:
        print("Failed JSON:", path, "|", repr(e))
        return None


def read_text(path: Path) -> str:
    if not path.exists():
        return ""

    try:
        return path.read_text(encoding="utf-8", errors="ignore")
    except Exception:
        return ""


def get_exp_dir(
    dataset_name: str,
    model_name: str,
    scenario_name: str,
    run_id: int,
) -> Path:
    return (
        RUN_DIR
        / safe_name(dataset_name)
        / safe_name(model_name)
        / safe_name(scenario_name)
        / f"run_{run_id}"
    )


def parse_log_best(log_text: str) -> Dict[str, Any]:
    out = {}

    patterns = {
        "log_best_epoch": r"BEST EPOCH\s*:\s*([0-9]+)",
        "log_best_val_macro_f1": r"BEST VAL MACRO F1\s*:\s*([0-9eE\.\+\-]+)",
        "log_best_val_acc": r"BEST VAL ACC\s*:\s*([0-9eE\.\+\-]+)",
        "log_best_val_loss": r"BEST VAL LOSS\s*:\s*([0-9eE\.\+\-]+)",
        "log_total_run_time_sec": r"TOTAL RUN TIME\s*:\s*([0-9eE\.\+\-]+)",
    }

    for k, pat in patterns.items():
        m = re.search(pat, log_text)
        if m:
            val = m.group(1)
            try:
                out[k] = int(val) if "epoch" in k else float(val)
            except Exception:
                out[k] = val

    return out


def get_history_metrics(history: Optional[Dict[str, Any]]) -> Dict[str, Any]:
    if not history:
        return {
            "n_epochs_recorded": 0,
            "best_epoch": None,
            "best_val_macro_f1": None,
            "best_val_accuracy": None,
            "best_val_loss": None,
            "best_val_weighted_f1": None,
            "best_val_micro_f1": None,
            "best_val_confidence": None,
            "final_val_macro_f1": None,
            "final_val_accuracy": None,
            "final_val_loss": None,
            "final_val_weighted_f1": None,
            "final_val_micro_f1": None,
            "final_val_confidence": None,
            "final_train_macro_f1": None,
            "final_train_accuracy": None,
            "final_train_loss": None,
            "mean_grad_norm": None,
            "mean_fractional_applied_params": None,
            "mean_epoch_time": None,
            "total_epoch_time": None,
            "mean_throughput": None,
        }

    val_macro_f1 = history.get("val_macro_f1", [])
    n_epochs = len(val_macro_f1) if isinstance(val_macro_f1, list) else 0

    best_idx = None
    if n_epochs > 0:
        s = pd.to_numeric(pd.Series(val_macro_f1), errors="coerce")
        if s.notna().sum() > 0:
            best_idx = int(s.idxmax())

    def at_best(key):
        values = history.get(key, [])
        if best_idx is not None and isinstance(values, list) and best_idx < len(values):
            return values[best_idx]
        return None

    def last(key):
        values = history.get(key, [])
        if isinstance(values, list) and len(values) > 0:
            return values[-1]
        return None

    def mean(key):
        values = history.get(key, [])
        if isinstance(values, list) and len(values) > 0:
            nums = pd.to_numeric(pd.Series(values), errors="coerce")
            if nums.notna().sum() > 0:
                return float(nums.mean())
        return None

    def total(key):
        values = history.get(key, [])
        if isinstance(values, list) and len(values) > 0:
            nums = pd.to_numeric(pd.Series(values), errors="coerce")
            if nums.notna().sum() > 0:
                return float(nums.sum())
        return None

    return {
        "n_epochs_recorded": n_epochs,

        "best_epoch": best_idx + 1 if best_idx is not None else None,
        "best_val_macro_f1": at_best("val_macro_f1"),
        "best_val_accuracy": at_best("val_accuracy"),
        "best_val_loss": at_best("val_loss"),
        "best_val_weighted_f1": at_best("val_weighted_f1"),
        "best_val_micro_f1": at_best("val_micro_f1"),
        "best_val_confidence": at_best("val_confidence"),

        "final_val_macro_f1": last("val_macro_f1"),
        "final_val_accuracy": last("val_accuracy"),
        "final_val_loss": last("val_loss"),
        "final_val_weighted_f1": last("val_weighted_f1"),
        "final_val_micro_f1": last("val_micro_f1"),
        "final_val_confidence": last("val_confidence"),

        "final_train_macro_f1": last("train_macro_f1"),
        "final_train_accuracy": last("train_accuracy"),
        "final_train_loss": last("train_loss"),

        "mean_grad_norm": mean("grad_norm"),
        "mean_fractional_applied_params": mean("fractional_applied_params"),
        "mean_epoch_time": mean("epoch_time"),
        "total_epoch_time": total("epoch_time"),
        "mean_throughput": mean("throughput_samples_per_sec"),
    }


def is_complete(exp_dir: Path, expected_epochs: int = EXPECTED_EPOCHS) -> bool:
    history_path = exp_dir / "history.json"
    best_model_path = exp_dir / "best_model.pt"
    final_model_path = exp_dir / "final_model.pt"

    if not history_path.exists():
        return False
    if not best_model_path.exists():
        return False
    if not final_model_path.exists():
        return False

    history = read_json(history_path)
    if not history:
        return False

    vals = history.get("val_macro_f1")
    if not isinstance(vals, list):
        return False

    return len(vals) >= expected_epochs


def flatten_cols(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df.columns = [
        "_".join([str(x) for x in col if str(x) != ""])
        if isinstance(col, tuple)
        else str(col)
        for col in df.columns
    ]
    return df


# ============================================================
# BUILD EXPECTED JOB TABLE
# ============================================================

rows = []
epoch_rows = []
file_rows = []

for dcfg in DATASET_CONFIGS:
    for mcfg in MODEL_CONFIGS:
        for scfg in SCENARIO_CONFIGS:
            for run_id in range(RUNS):
                dataset_name = dcfg["dataset_name"]
                model_name = mcfg["model_name"]
                scenario_name = scfg["scenario_name"]

                exp_dir = get_exp_dir(
                    dataset_name=dataset_name,
                    model_name=model_name,
                    scenario_name=scenario_name,
                    run_id=run_id,
                )

                history_path = exp_dir / "history.json"
                log_path = exp_dir / "log.txt"
                initial_model_path = exp_dir / "initial_model.pt"
                best_model_path = exp_dir / "best_model.pt"
                final_model_path = exp_dir / "final_model.pt"
                dataset_info_path = exp_dir / "dataset_info.json"
                model_config_path = exp_dir / "model_config.json"
                scenario_config_path = exp_dir / "scenario_config.json"

                history = read_json(history_path)
                log_text = read_text(log_path)

                metrics = get_history_metrics(history)
                log_metrics = parse_log_best(log_text)

                complete = is_complete(exp_dir)

                if complete:
                    status = "complete"
                elif exp_dir.exists():
                    status = "incomplete_or_broken"
                else:
                    status = "missing"

                row = {
                    "status": status,
                    "is_complete": complete,

                    "dataset_name": dataset_name,
                    "n_samples": dcfg["n_samples"],
                    "depth": dcfg["depth"],
                    "max_len": dcfg["max_len"],

                    "model_name": model_name,
                    "d_model": mcfg["d_model"],
                    "nhead": mcfg["nhead"],
                    "num_layers": mcfg["num_layers"],
                    "dim_feedforward": mcfg["dim_feedforward"],
                    "dropout": mcfg["dropout"],
                    "pooling": mcfg["pooling"],

                    "scenario_name": scenario_name,
                    "base_optimizer": scfg.get("base_optimizer"),
                    "lr": scfg.get("lr"),
                    "weight_decay": scfg.get("weight_decay"),
                    "mode": scfg.get("mode"),
                    "target": scfg.get("target"),
                    "alpha": scfg.get("alpha"),
                    "beta": scfg.get("beta"),
                    "mix_lambda": scfg.get("mix_lambda"),
                    "start_epoch": scfg.get("start_epoch"),

                    "run_id": run_id,

                    "exp_dir": str(exp_dir),
                    "has_exp_dir": exp_dir.exists(),
                    "has_history_json": history_path.exists(),
                    "has_log_txt": log_path.exists(),
                    "has_initial_model_pt": initial_model_path.exists(),
                    "has_best_model_pt": best_model_path.exists(),
                    "has_final_model_pt": final_model_path.exists(),
                    "has_dataset_info_json": dataset_info_path.exists(),
                    "has_model_config_json": model_config_path.exists(),
                    "has_scenario_config_json": scenario_config_path.exists(),

                    "history_path": str(history_path),
                    "log_path": str(log_path),
                    "best_model_path": str(best_model_path),
                    "final_model_path": str(final_model_path),
                }

                row.update(metrics)
                row.update(log_metrics)

                rows.append(row)

                # epoch rows
                if isinstance(history, dict) and len(history) > 0:
                    max_epochs = max(
                        [len(v) for v in history.values() if isinstance(v, list)] or [0]
                    )

                    for eidx in range(max_epochs):
                        erow = {
                            "dataset_name": dataset_name,
                            "model_name": model_name,
                            "scenario_name": scenario_name,
                            "run_id": run_id,
                            "epoch": eidx + 1,

                            "mode": scfg.get("mode"),
                            "target": scfg.get("target"),
                            "alpha": scfg.get("alpha"),
                            "beta": scfg.get("beta"),
                            "mix_lambda": scfg.get("mix_lambda"),
                            "start_epoch": scfg.get("start_epoch"),

                            "d_model": mcfg["d_model"],
                            "num_layers": mcfg["num_layers"],
                            "pooling": mcfg["pooling"],
                        }

                        for key, values in history.items():
                            if isinstance(values, list):
                                erow[key] = values[eidx] if eidx < len(values) else None

                        epoch_rows.append(erow)

                # file rows
                for f in [
                    history_path,
                    log_path,
                    initial_model_path,
                    best_model_path,
                    final_model_path,
                    dataset_info_path,
                    model_config_path,
                    scenario_config_path,
                ]:
                    file_rows.append(
                        {
                            "dataset_name": dataset_name,
                            "model_name": model_name,
                            "scenario_name": scenario_name,
                            "run_id": run_id,
                            "file_name": f.name,
                            "file_path": str(f),
                            "exists": f.exists(),
                            "size_bytes": f.stat().st_size if f.exists() else None,
                        }
                    )

all_jobs_df = pd.DataFrame(rows)
epochs_df = pd.DataFrame(epoch_rows)
files_df = pd.DataFrame(file_rows)

# numeric cleanup
for df in [all_jobs_df, epochs_df]:
    if not df.empty:
        for c in df.columns:
            if c not in [
                "status",
                "dataset_name",
                "model_name",
                "scenario_name",
                "base_optimizer",
                "mode",
                "target",
                "pooling",
                "exp_dir",
                "history_path",
                "log_path",
                "best_model_path",
                "final_model_path",
            ]:
                # ✅ CHANGED: "ignore" -> "coerce"
                df[c] = pd.to_numeric(df[c], errors="coerce")


print("\nStatus counts:")
print(all_jobs_df["status"].value_counts(dropna=False))
print("\nall_jobs_df:", all_jobs_df.shape)
print("epochs_df  :", epochs_df.shape)
print("files_df   :", files_df.shape)


# ============================================================
# OVERVIEW
# ============================================================

total_expected = len(all_jobs_df)
n_complete = int((all_jobs_df["status"] == "complete").sum())
n_incomplete = int((all_jobs_df["status"] == "incomplete_or_broken").sum())
n_missing = int((all_jobs_df["status"] == "missing").sum())
recorded_epochs = int(pd.to_numeric(all_jobs_df["n_epochs_recorded"], errors="coerce").fillna(0).sum())

overview_df = pd.DataFrame(
    [
        {"metric": "expected_total_jobs", "value": total_expected},
        {"metric": "complete_jobs", "value": n_complete},
        {"metric": "incomplete_or_broken_jobs", "value": n_incomplete},
        {"metric": "missing_jobs", "value": n_missing},
        {"metric": "completion_percent", "value": 100.0 * n_complete / max(total_expected, 1)},
        {"metric": "expected_epochs_total", "value": total_expected * EXPECTED_EPOCHS},
        {"metric": "recorded_epochs_total", "value": recorded_epochs},
        {"metric": "recorded_epochs_percent", "value": 100.0 * recorded_epochs / max(total_expected * EXPECTED_EPOCHS, 1)},
    ]
)


# ============================================================
# TABLES
# ============================================================

complete_df = all_jobs_df[all_jobs_df["status"] == "complete"].copy()
incomplete_df = all_jobs_df[all_jobs_df["status"] == "incomplete_or_broken"].copy()
missing_df = all_jobs_df[all_jobs_df["status"] == "missing"].copy()

leaderboard_df = (
    complete_df
    .sort_values("best_val_macro_f1", ascending=False)
    .reset_index(drop=True)
)

best_by_combo_df = (
    complete_df
    .sort_values("best_val_macro_f1", ascending=False)
    .groupby(["dataset_name", "model_name", "scenario_name"], as_index=False)
    .head(1)
    .reset_index(drop=True)
)

agg_metrics = [
    "best_val_macro_f1",
    "best_val_accuracy",
    "best_val_loss",
    "final_val_macro_f1",
    "final_val_accuracy",
    "final_val_loss",
    "mean_grad_norm",
    "mean_fractional_applied_params",
    "mean_epoch_time",
    "mean_throughput",
]

available_agg_metrics = [c for c in agg_metrics if c in complete_df.columns]

by_scenario_df = (
    complete_df
    .groupby(
        [
            "scenario_name",
            "mode",
            "target",
            "alpha",
            "beta",
            "mix_lambda",
            "start_epoch",
        ],
        dropna=False,
    )[available_agg_metrics]
    .agg(["count", "mean", "std", "min", "max"])
    .reset_index()
)

by_scenario_df = flatten_cols(by_scenario_df)

if "best_val_macro_f1_mean" in by_scenario_df.columns:
    by_scenario_df = by_scenario_df.sort_values(
        "best_val_macro_f1_mean",
        ascending=False,
    ).reset_index(drop=True)


by_dataset_scenario_df = (
    complete_df
    .groupby(
        [
            "dataset_name",
            "scenario_name",
            "mode",
            "target",
        ],
        dropna=False,
    )[available_agg_metrics]
    .agg(["count", "mean", "std", "min", "max"])
    .reset_index()
)

by_dataset_scenario_df = flatten_cols(by_dataset_scenario_df)

if "best_val_macro_f1_mean" in by_dataset_scenario_df.columns:
    by_dataset_scenario_df = by_dataset_scenario_df.sort_values(
        ["dataset_name", "best_val_macro_f1_mean"],
        ascending=[True, False],
    ).reset_index(drop=True)


by_model_scenario_df = (
    complete_df
    .groupby(
        [
            "model_name",
            "scenario_name",
            "mode",
            "target",
        ],
        dropna=False,
    )[available_agg_metrics]
    .agg(["count", "mean", "std", "min", "max"])
    .reset_index()
)

by_model_scenario_df = flatten_cols(by_model_scenario_df)

if "best_val_macro_f1_mean" in by_model_scenario_df.columns:
    by_model_scenario_df = by_model_scenario_df.sort_values(
        ["model_name", "best_val_macro_f1_mean"],
        ascending=[True, False],
    ).reset_index(drop=True)


by_dataset_model_scenario_df = (
    complete_df
    .groupby(
        [
            "dataset_name",
            "model_name",
            "scenario_name",
            "mode",
            "target",
        ],
        dropna=False,
    )[available_agg_metrics]
    .agg(["count", "mean", "std", "min", "max"])
    .reset_index()
)

by_dataset_model_scenario_df = flatten_cols(by_dataset_model_scenario_df)

if "best_val_macro_f1_mean" in by_dataset_model_scenario_df.columns:
    by_dataset_model_scenario_df = by_dataset_model_scenario_df.sort_values(
        ["dataset_name", "model_name", "best_val_macro_f1_mean"],
        ascending=[True, True, False],
    ).reset_index(drop=True)


# ============================================================
# BASELINE COMPARISON
# ============================================================

baseline_df = complete_df[complete_df["scenario_name"] == "baseline_adamw"].copy()

baseline_keys = ["dataset_name", "model_name", "run_id"]

baseline_ref = baseline_df[
    baseline_keys
    + [
        "best_val_macro_f1",
        "best_val_accuracy",
        "best_val_loss",
        "final_val_macro_f1",
        "final_val_accuracy",
        "final_val_loss",
    ]
].rename(
    columns={
        "best_val_macro_f1": "baseline_best_val_macro_f1",
        "best_val_accuracy": "baseline_best_val_accuracy",
        "best_val_loss": "baseline_best_val_loss",
        "final_val_macro_f1": "baseline_final_val_macro_f1",
        "final_val_accuracy": "baseline_final_val_accuracy",
        "final_val_loss": "baseline_final_val_loss",
    }
)

comparison_df = complete_df.merge(
    baseline_ref,
    on=baseline_keys,
    how="left",
)

comparison_df["delta_best_val_macro_f1_vs_baseline"] = (
    comparison_df["best_val_macro_f1"] - comparison_df["baseline_best_val_macro_f1"]
)

comparison_df["delta_best_val_accuracy_vs_baseline"] = (
    comparison_df["best_val_accuracy"] - comparison_df["baseline_best_val_accuracy"]
)

comparison_df["delta_final_val_macro_f1_vs_baseline"] = (
    comparison_df["final_val_macro_f1"] - comparison_df["baseline_final_val_macro_f1"]
)

comparison_df["beats_baseline_best_macro_f1"] = (
    comparison_df["delta_best_val_macro_f1_vs_baseline"] > 0
)

scenario_vs_baseline_df = (
    comparison_df[comparison_df["scenario_name"] != "baseline_adamw"]
    .groupby(
        [
            "scenario_name",
            "mode",
            "target",
            "alpha",
            "beta",
            "mix_lambda",
            "start_epoch",
        ],
        dropna=False,
    )[
        [
            "delta_best_val_macro_f1_vs_baseline",
            "delta_best_val_accuracy_vs_baseline",
            "delta_final_val_macro_f1_vs_baseline",
            "beats_baseline_best_macro_f1",
        ]
    ]
    .agg(["count", "mean", "std", "min", "max", "sum"])
    .reset_index()
)

scenario_vs_baseline_df = flatten_cols(scenario_vs_baseline_df)

if "delta_best_val_macro_f1_vs_baseline_mean" in scenario_vs_baseline_df.columns:
    scenario_vs_baseline_df = scenario_vs_baseline_df.sort_values(
        "delta_best_val_macro_f1_vs_baseline_mean",
        ascending=False,
    ).reset_index(drop=True)


# ============================================================
# CURRENT PROGRESS BY BLOCK
# ============================================================

progress_by_dataset_df = (
    all_jobs_df
    .groupby(["dataset_name"], dropna=False)["status"]
    .value_counts()
    .unstack(fill_value=0)
    .reset_index()
)

progress_by_model_df = (
    all_jobs_df
    .groupby(["model_name"], dropna=False)["status"]
    .value_counts()
    .unstack(fill_value=0)
    .reset_index()
)

progress_by_scenario_df = (
    all_jobs_df
    .groupby(["scenario_name"], dropna=False)["status"]
    .value_counts()
    .unstack(fill_value=0)
    .reset_index()
)


# ============================================================
# SAVE
# ============================================================

all_jobs_df.to_csv(OUT_CSV, index=False)

with pd.ExcelWriter(OUT_XLSX, engine="openpyxl") as writer:
    overview_df.to_excel(writer, sheet_name="overview", index=False)

    all_jobs_df.to_excel(writer, sheet_name="all_expected_jobs", index=False)
    complete_df.to_excel(writer, sheet_name="complete_runs", index=False)
    incomplete_df.to_excel(writer, sheet_name="incomplete_runs", index=False)
    missing_df.to_excel(writer, sheet_name="missing_runs", index=False)

    leaderboard_df.to_excel(writer, sheet_name="leaderboard", index=False)
    best_by_combo_df.to_excel(writer, sheet_name="best_by_combo", index=False)

    by_scenario_df.to_excel(writer, sheet_name="by_scenario", index=False)
    by_dataset_scenario_df.to_excel(writer, sheet_name="by_dataset_scenario", index=False)
    by_model_scenario_df.to_excel(writer, sheet_name="by_model_scenario", index=False)
    by_dataset_model_scenario_df.to_excel(writer, sheet_name="by_dataset_model_scenario", index=False)

    comparison_df.to_excel(writer, sheet_name="comparison_vs_baseline", index=False)
    scenario_vs_baseline_df.to_excel(writer, sheet_name="scenario_vs_baseline", index=False)

    progress_by_dataset_df.to_excel(writer, sheet_name="progress_by_dataset", index=False)
    progress_by_model_df.to_excel(writer, sheet_name="progress_by_model", index=False)
    progress_by_scenario_df.to_excel(writer, sheet_name="progress_by_scenario", index=False)

    epochs_df.to_excel(writer, sheet_name="epochs_all", index=False)
    files_df.to_excel(writer, sheet_name="files_index", index=False)

print("\nDONE")
print("CSV saved:", OUT_CSV)
print("Excel saved:", OUT_XLSX)

print("\nOverview:")
print(overview_df)

print("\nTop 15 leaderboard:")
cols = [
    "dataset_name",
    "model_name",
    "scenario_name",
    "run_id",
    "best_val_macro_f1",
    "best_val_accuracy",
    "best_val_loss",
    "best_epoch",
]
print(leaderboard_df[cols].head(15))

print("\nScenario vs baseline:")
cols2 = [
    "scenario_name",
    "delta_best_val_macro_f1_vs_baseline_count",
    "delta_best_val_macro_f1_vs_baseline_mean",
    "delta_best_val_macro_f1_vs_baseline_std",
    "delta_best_val_macro_f1_vs_baseline_min",
    "delta_best_val_macro_f1_vs_baseline_max",
    "beats_baseline_best_macro_f1_mean",
    "beats_baseline_best_macro_f1_sum",
]
cols2 = [c for c in cols2 if c in scenario_vs_baseline_df.columns]
print(scenario_vs_baseline_df[cols2])

ROOT: /home/tahiti/Malashin_Projects
RUN_DIR: /home/tahiti/Malashin_Projects/runs_fractional_scenarios
OUT_XLSX: /home/tahiti/Malashin_Projects/fractional_scenarios_statistics.xlsx

Status counts:
status
complete    495
Name: count, dtype: int64

all_jobs_df: (495, 65)
epochs_df  : (4950, 35)
files_df   : (3960, 8)

DONE
CSV saved: /home/tahiti/Malashin_Projects/fractional_scenarios_all_jobs.csv
Excel saved: /home/tahiti/Malashin_Projects/fractional_scenarios_statistics.xlsx

Overview:
                      metric   value
0        expected_total_jobs   495.0
1              complete_jobs   495.0
2  incomplete_or_broken_jobs     0.0
3               missing_jobs     0.0
4         completion_percent   100.0
5      expected_epochs_total  4950.0
6      recorded_epochs_total  4950.0
7    recorded_epochs_percent   100.0

Top 15 leaderboard:
    dataset_name    model_name                        scenario_name  run_id  \
0   small_D2_L64   T_deep_mean                       baseline_adamw       1 

In [4]:
from __future__ import annotations

import json
import re
from pathlib import Path
from typing import Any, Dict, Optional

import pandas as pd


# ============================================================
# CONFIG
# ============================================================

ROOT = Path.cwd()

RUN_DIRS = [
    ROOT / "runs_transformer_fractional",
    ROOT / "runs_fractional_scenarios",
    ROOT / "runs_fractional_hypothesis_v2",
    ROOT / "runs_fractional_focused_final",
]

OUT_XLSX = ROOT / "ALL_fractional_experiments_statistics.xlsx"
OUT_CSV = ROOT / "ALL_fractional_experiments_all_runs.csv"

EXPECTED_EPOCHS_DEFAULT = 10

print("ROOT:", ROOT)
print("Output:", OUT_XLSX)


# ============================================================
# HELPERS
# ============================================================

def read_json(path: Path) -> Optional[Dict[str, Any]]:
    if not path.exists():
        return None
    try:
        return json.loads(path.read_text(encoding="utf-8"))
    except Exception:
        return None


def read_text(path: Path) -> str:
    if not path.exists():
        return ""
    try:
        return path.read_text(encoding="utf-8", errors="ignore")
    except Exception:
        return ""


def parse_log_best(text: str) -> Dict[str, Any]:
    out = {}
    patterns = {
        "log_best_epoch": r"BEST EPOCH\s*:\s*([0-9]+)",
        "log_best_val_macro_f1": r"BEST VAL MACRO F1\s*:\s*([0-9eE\.\+\-]+)",
        "log_best_val_acc": r"BEST VAL ACC\s*:\s*([0-9eE\.\+\-]+)",
        "log_best_val_loss": r"BEST VAL LOSS\s*:\s*([0-9eE\.\+\-]+)",
        "log_total_run_time_sec": r"TOTAL RUN TIME\s*:\s*([0-9eE\.\+\-]+)",
    }

    for k, pat in patterns.items():
        m = re.search(pat, text)
        if m:
            try:
                out[k] = int(m.group(1)) if "epoch" in k else float(m.group(1))
            except Exception:
                out[k] = m.group(1)
    return out


def history_metrics(history: Optional[Dict[str, Any]]) -> Dict[str, Any]:
    if not history:
        return {"n_epochs_recorded": 0}

    val_f1 = history.get("val_macro_f1", [])
    n_epochs = len(val_f1) if isinstance(val_f1, list) else 0

    best_idx = None
    if n_epochs > 0:
        s = pd.to_numeric(pd.Series(val_f1), errors="coerce")
        if s.notna().any():
            best_idx = int(s.idxmax())

    def at_best(key):
        vals = history.get(key, [])
        if best_idx is not None and isinstance(vals, list) and best_idx < len(vals):
            return vals[best_idx]
        return None

    def last(key):
        vals = history.get(key, [])
        if isinstance(vals, list) and vals:
            return vals[-1]
        return None

    def mean(key):
        vals = history.get(key, [])
        if isinstance(vals, list) and vals:
            s = pd.to_numeric(pd.Series(vals), errors="coerce")
            return float(s.mean()) if s.notna().any() else None
        return None

    def total(key):
        vals = history.get(key, [])
        if isinstance(vals, list) and vals:
            s = pd.to_numeric(pd.Series(vals), errors="coerce")
            return float(s.sum()) if s.notna().any() else None
        return None

    return {
        "n_epochs_recorded": n_epochs,
        "best_epoch": best_idx + 1 if best_idx is not None else None,

        "best_val_macro_f1": at_best("val_macro_f1"),
        "best_val_accuracy": at_best("val_accuracy"),
        "best_val_loss": at_best("val_loss"),
        "best_val_micro_f1": at_best("val_micro_f1"),
        "best_val_weighted_f1": at_best("val_weighted_f1"),
        "best_val_confidence": at_best("val_confidence"),

        "final_val_macro_f1": last("val_macro_f1"),
        "final_val_accuracy": last("val_accuracy"),
        "final_val_loss": last("val_loss"),
        "final_val_micro_f1": last("val_micro_f1"),
        "final_val_weighted_f1": last("val_weighted_f1"),
        "final_val_confidence": last("val_confidence"),

        "final_train_macro_f1": last("train_macro_f1"),
        "final_train_accuracy": last("train_accuracy"),
        "final_train_loss": last("train_loss"),

        "mean_grad_norm": mean("grad_norm"),
        "mean_fractional_applied_params": mean("fractional_applied_params"),
        "mean_epoch_time": mean("epoch_time"),
        "total_epoch_time": total("epoch_time"),
        "mean_throughput": mean("throughput_samples_per_sec"),
    }


def flatten_cols(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df.columns = [
        "_".join([str(x) for x in col if str(x) != ""])
        if isinstance(col, tuple)
        else str(col)
        for col in df.columns
    ]
    return df


# ============================================================
# COLLECT RUNS
# ============================================================

run_rows = []
epoch_rows = []
file_rows = []

for run_root in RUN_DIRS:
    if not run_root.exists():
        print("Missing dir:", run_root)
        continue

    experiment_name = run_root.name
    print("Scanning:", run_root)

    for exp_dir in sorted(run_root.glob("*/*/*/run_*")):
        if not exp_dir.is_dir():
            continue

        parts = exp_dir.relative_to(run_root).parts
        if len(parts) < 4:
            continue

        dataset_name = parts[0]
        model_name = parts[1]
        scenario_or_optimizer = parts[2]
        run_name = parts[3]

        try:
            run_id = int(run_name.replace("run_", ""))
        except Exception:
            run_id = None

        history_path = exp_dir / "history.json"
        log_path = exp_dir / "log.txt"
        dataset_info_path = exp_dir / "dataset_info.json"
        model_config_path = exp_dir / "model_config.json"

        scenario_config_path = exp_dir / "scenario_config.json"
        optimizer_config_path = exp_dir / "optimizer_config.json"

        initial_model_path = exp_dir / "initial_model.pt"
        best_model_path = exp_dir / "best_model.pt"
        final_model_path = exp_dir / "final_model.pt"

        history = read_json(history_path)
        dataset_info = read_json(dataset_info_path) or {}
        model_cfg = read_json(model_config_path) or {}
        scenario_cfg = read_json(scenario_config_path) or {}
        optimizer_cfg = read_json(optimizer_config_path) or {}

        cfg = {}
        cfg.update(optimizer_cfg)
        cfg.update(scenario_cfg)

        log_text = read_text(log_path)

        metrics = history_metrics(history)
        log_metrics = parse_log_best(log_text)

        complete = (
            history_path.exists()
            and best_model_path.exists()
            and final_model_path.exists()
            and metrics.get("n_epochs_recorded", 0) >= EXPECTED_EPOCHS_DEFAULT
        )

        if complete:
            status = "complete"
        elif exp_dir.exists():
            status = "incomplete_or_broken"
        else:
            status = "missing"

        row = {
            "experiment_name": experiment_name,
            "status": status,
            "is_complete": complete,

            "dataset_name": dataset_info.get("dataset_name", dataset_name),
            "model_name": model_cfg.get("model_name", model_name),

            "scenario_name": cfg.get("scenario_name", scenario_or_optimizer),
            "optimizer_name": cfg.get("optimizer_name", cfg.get("scenario_name", scenario_or_optimizer)),
            "base_optimizer": cfg.get("base_optimizer"),

            "run_id": run_id,

            "n_samples": dataset_info.get("n_samples"),
            "seq_len": dataset_info.get("seq_len"),
            "vocab_size": dataset_info.get("vocab_size"),
            "num_classes": dataset_info.get("num_classes"),
            "mean_length": dataset_info.get("mean_length"),
            "max_length": dataset_info.get("max_length"),

            "d_model": model_cfg.get("d_model"),
            "nhead": model_cfg.get("nhead"),
            "num_layers": model_cfg.get("num_layers"),
            "dim_feedforward": model_cfg.get("dim_feedforward"),
            "dropout": model_cfg.get("dropout"),
            "pooling": model_cfg.get("pooling"),

            "mode": cfg.get("mode"),
            "target": cfg.get("target"),
            "alpha": cfg.get("alpha", cfg.get("fractional_alpha")),
            "beta": cfg.get("beta", cfg.get("fractional_beta")),
            "mix_lambda": cfg.get("mix_lambda"),
            "start_epoch": cfg.get("start_epoch"),

            "lr": cfg.get("lr"),
            "weight_decay": cfg.get("weight_decay"),
            "fractional": cfg.get("fractional"),

            "exp_dir": str(exp_dir),
            "history_path": str(history_path),
            "log_path": str(log_path),
            "initial_model_path": str(initial_model_path),
            "best_model_path": str(best_model_path),
            "final_model_path": str(final_model_path),

            "has_history_json": history_path.exists(),
            "has_log_txt": log_path.exists(),
            "has_initial_model_pt": initial_model_path.exists(),
            "has_best_model_pt": best_model_path.exists(),
            "has_final_model_pt": final_model_path.exists(),
        }

        row.update(metrics)
        row.update(log_metrics)

        run_rows.append(row)

        if isinstance(history, dict):
            max_epochs = max([len(v) for v in history.values() if isinstance(v, list)] or [0])

            for eidx in range(max_epochs):
                erow = {
                    "experiment_name": experiment_name,
                    "dataset_name": row["dataset_name"],
                    "model_name": row["model_name"],
                    "scenario_name": row["scenario_name"],
                    "optimizer_name": row["optimizer_name"],
                    "run_id": run_id,
                    "epoch": eidx + 1,
                    "mode": row["mode"],
                    "target": row["target"],
                    "alpha": row["alpha"],
                    "beta": row["beta"],
                    "mix_lambda": row["mix_lambda"],
                    "start_epoch": row["start_epoch"],
                }

                for key, vals in history.items():
                    if isinstance(vals, list):
                        erow[key] = vals[eidx] if eidx < len(vals) else None

                epoch_rows.append(erow)

        for f in [
            history_path,
            log_path,
            dataset_info_path,
            model_config_path,
            scenario_config_path,
            optimizer_config_path,
            initial_model_path,
            best_model_path,
            final_model_path,
        ]:
            file_rows.append(
                {
                    "experiment_name": experiment_name,
                    "dataset_name": row["dataset_name"],
                    "model_name": row["model_name"],
                    "scenario_name": row["scenario_name"],
                    "run_id": run_id,
                    "file_name": f.name,
                    "file_path": str(f),
                    "exists": f.exists(),
                    "size_bytes": f.stat().st_size if f.exists() else None,
                }
            )


all_runs_df = pd.DataFrame(run_rows)
epochs_df = pd.DataFrame(epoch_rows)
files_df = pd.DataFrame(file_rows)

print("\nCollected:")
print("all_runs_df:", all_runs_df.shape)
print("epochs_df  :", epochs_df.shape)
print("files_df   :", files_df.shape)

if all_runs_df.empty:
    raise RuntimeError("No runs found.")

# ============================================================
# NUMERIC CLEANUP
# ============================================================

for df in [all_runs_df, epochs_df]:
    for c in df.columns:
        if c not in {
            "experiment_name",
            "status",
            "dataset_name",
            "model_name",
            "scenario_name",
            "optimizer_name",
            "base_optimizer",
            "mode",
            "target",
            "pooling",
            "exp_dir",
            "history_path",
            "log_path",
            "initial_model_path",
            "best_model_path",
            "final_model_path",
        }:
            # Changed errors="ignore" to errors="coerce"
            df[c] = pd.to_numeric(df[c], errors="coerce")


# ============================================================
# OVERVIEW
# ============================================================

overview_df = (
    all_runs_df
    .groupby(["experiment_name", "status"], dropna=False)
    .size()
    .reset_index(name="count")
)

overview_total_df = pd.DataFrame(
    [
        {"metric": "total_runs_found", "value": len(all_runs_df)},
        {"metric": "complete_runs", "value": int((all_runs_df["status"] == "complete").sum())},
        {"metric": "incomplete_or_broken_runs", "value": int((all_runs_df["status"] == "incomplete_or_broken").sum())},
        {"metric": "total_epochs_recorded", "value": int(pd.to_numeric(all_runs_df["n_epochs_recorded"], errors="coerce").fillna(0).sum())},
    ]
)


complete_df = all_runs_df[all_runs_df["status"] == "complete"].copy()
broken_df = all_runs_df[all_runs_df["status"] != "complete"].copy()

leaderboard_df = (
    complete_df
    .sort_values("best_val_macro_f1", ascending=False)
    .reset_index(drop=True)
)


# ============================================================
# AGGREGATIONS
# ============================================================

metrics = [
    "best_val_macro_f1",
    "best_val_accuracy",
    "best_val_loss",
    "final_val_macro_f1",
    "final_val_accuracy",
    "final_val_loss",
    "mean_grad_norm",
    "mean_fractional_applied_params",
    "mean_epoch_time",
    "mean_throughput",
    "total_epoch_time",
]

metrics = [m for m in metrics if m in complete_df.columns]

by_experiment_df = flatten_cols(
    complete_df
    .groupby(["experiment_name"], dropna=False)[metrics]
    .agg(["count", "mean", "std", "min", "max"])
    .reset_index()
)

by_scenario_df = flatten_cols(
    complete_df
    .groupby(
        [
            "experiment_name",
            "scenario_name",
            "mode",
            "target",
            "alpha",
            "beta",
            "mix_lambda",
            "start_epoch",
        ],
        dropna=False,
    )[metrics]
    .agg(["count", "mean", "std", "min", "max"])
    .reset_index()
)

by_dataset_scenario_df = flatten_cols(
    complete_df
    .groupby(
        [
            "experiment_name",
            "dataset_name",
            "scenario_name",
            "mode",
            "target",
            "alpha",
            "beta",
            "mix_lambda",
            "start_epoch",
        ],
        dropna=False,
    )[metrics]
    .agg(["count", "mean", "std", "min", "max"])
    .reset_index()
)

by_model_scenario_df = flatten_cols(
    complete_df
    .groupby(
        [
            "experiment_name",
            "model_name",
            "scenario_name",
            "mode",
            "target",
            "alpha",
            "beta",
            "mix_lambda",
            "start_epoch",
        ],
        dropna=False,
    )[metrics]
    .agg(["count", "mean", "std", "min", "max"])
    .reset_index()
)

by_dataset_model_scenario_df = flatten_cols(
    complete_df
    .groupby(
        [
            "experiment_name",
            "dataset_name",
            "model_name",
            "scenario_name",
            "mode",
            "target",
            "alpha",
            "beta",
            "mix_lambda",
            "start_epoch",
        ],
        dropna=False,
    )[metrics]
    .agg(["count", "mean", "std", "min", "max"])
    .reset_index()
)

for df in [
    by_experiment_df,
    by_scenario_df,
    by_dataset_scenario_df,
    by_model_scenario_df,
    by_dataset_model_scenario_df,
]:
    if "best_val_macro_f1_mean" in df.columns:
        df.sort_values("best_val_macro_f1_mean", ascending=False, inplace=True)


# ============================================================
# BASELINE COMPARISON
# ============================================================

baseline_df = complete_df[complete_df["scenario_name"].eq("baseline_adamw")].copy()

baseline_keys = ["experiment_name", "dataset_name", "model_name", "run_id"]

baseline_ref = baseline_df[
    baseline_keys
    + [
        "best_val_macro_f1",
        "best_val_accuracy",
        "best_val_loss",
        "final_val_macro_f1",
        "final_val_accuracy",
        "final_val_loss",
    ]
].rename(
    columns={
        "best_val_macro_f1": "baseline_best_val_macro_f1",
        "best_val_accuracy": "baseline_best_val_accuracy",
        "best_val_loss": "baseline_best_val_loss",
        "final_val_macro_f1": "baseline_final_val_macro_f1",
        "final_val_accuracy": "baseline_final_val_accuracy",
        "final_val_loss": "baseline_final_val_loss",
    }
)

comparison_df = complete_df.merge(
    baseline_ref,
    on=baseline_keys,
    how="left",
)

comparison_df["delta_best_val_macro_f1_vs_baseline"] = (
    comparison_df["best_val_macro_f1"] - comparison_df["baseline_best_val_macro_f1"]
)

comparison_df["delta_best_val_accuracy_vs_baseline"] = (
    comparison_df["best_val_accuracy"] - comparison_df["baseline_best_val_accuracy"]
)

comparison_df["delta_final_val_macro_f1_vs_baseline"] = (
    comparison_df["final_val_macro_f1"] - comparison_df["baseline_final_val_macro_f1"]
)

comparison_df["beats_baseline_best_macro_f1"] = (
    comparison_df["delta_best_val_macro_f1_vs_baseline"] > 0
)

scenario_vs_baseline_df = flatten_cols(
    comparison_df[comparison_df["scenario_name"] != "baseline_adamw"]
    .groupby(
        [
            "experiment_name",
            "scenario_name",
            "mode",
            "target",
            "alpha",
            "beta",
            "mix_lambda",
            "start_epoch",
        ],
        dropna=False,
    )[
        [
            "delta_best_val_macro_f1_vs_baseline",
            "delta_best_val_accuracy_vs_baseline",
            "delta_final_val_macro_f1_vs_baseline",
            "beats_baseline_best_macro_f1",
        ]
    ]
    .agg(["count", "mean", "std", "min", "max", "sum"])
    .reset_index()
)

if "delta_best_val_macro_f1_vs_baseline_mean" in scenario_vs_baseline_df.columns:
    scenario_vs_baseline_df.sort_values(
        "delta_best_val_macro_f1_vs_baseline_mean",
        ascending=False,
        inplace=True,
    )


# ============================================================
# BEST BY GROUP
# ============================================================

best_by_dataset_df = (
    complete_df
    .sort_values("best_val_macro_f1", ascending=False)
    .groupby(["experiment_name", "dataset_name"], as_index=False)
    .head(1)
    .reset_index(drop=True)
)

best_by_model_df = (
    complete_df
    .sort_values("best_val_macro_f1", ascending=False)
    .groupby(["experiment_name", "model_name"], as_index=False)
    .head(1)
    .reset_index(drop=True)
)

best_by_dataset_model_df = (
    complete_df
    .sort_values("best_val_macro_f1", ascending=False)
    .groupby(["experiment_name", "dataset_name", "model_name"], as_index=False)
    .head(1)
    .reset_index(drop=True)
)


# ============================================================
# SAVE
# ============================================================

all_runs_df.to_csv(OUT_CSV, index=False)

with pd.ExcelWriter(OUT_XLSX, engine="openpyxl") as writer:
    overview_total_df.to_excel(writer, sheet_name="overview_total", index=False)
    overview_df.to_excel(writer, sheet_name="overview_by_experiment", index=False)

    all_runs_df.to_excel(writer, sheet_name="all_runs", index=False)
    complete_df.to_excel(writer, sheet_name="complete_runs", index=False)
    broken_df.to_excel(writer, sheet_name="broken_runs", index=False)

    leaderboard_df.to_excel(writer, sheet_name="leaderboard", index=False)

    by_experiment_df.to_excel(writer, sheet_name="by_experiment", index=False)
    by_scenario_df.to_excel(writer, sheet_name="by_scenario", index=False)
    by_dataset_scenario_df.to_excel(writer, sheet_name="by_dataset_scenario", index=False)
    by_model_scenario_df.to_excel(writer, sheet_name="by_model_scenario", index=False)
    by_dataset_model_scenario_df.to_excel(writer, sheet_name="by_dataset_model_scenario", index=False)

    comparison_df.to_excel(writer, sheet_name="comparison_vs_baseline", index=False)
    scenario_vs_baseline_df.to_excel(writer, sheet_name="scenario_vs_baseline", index=False)

    best_by_dataset_df.to_excel(writer, sheet_name="best_by_dataset", index=False)
    best_by_model_df.to_excel(writer, sheet_name="best_by_model", index=False)
    best_by_dataset_model_df.to_excel(writer, sheet_name="best_by_dataset_model", index=False)

    epochs_df.to_excel(writer, sheet_name="epochs_all", index=False)
    files_df.to_excel(writer, sheet_name="files_index", index=False)


print("\nDONE")
print("CSV:", OUT_CSV)
print("Excel:", OUT_XLSX)

print("\nOverview total:")
print(overview_total_df)

print("\nTop 20:")
show_cols = [
    "experiment_name",
    "dataset_name",
    "model_name",
    "scenario_name",
    "run_id",
    "best_val_macro_f1",
    "best_val_accuracy",
    "best_val_loss",
    "best_epoch",
]
print(leaderboard_df[show_cols].head(20))

print("\nScenario vs baseline top:")
show_cols2 = [
    "experiment_name",
    "scenario_name",
    "delta_best_val_macro_f1_vs_baseline_count",
    "delta_best_val_macro_f1_vs_baseline_mean",
    "delta_best_val_macro_f1_vs_baseline_std",
    "delta_best_val_macro_f1_vs_baseline_min",
    "delta_best_val_macro_f1_vs_baseline_max",
    "beats_baseline_best_macro_f1_mean",
    "beats_baseline_best_macro_f1_sum",
]
show_cols2 = [c for c in show_cols2 if c in scenario_vs_baseline_df.columns]
print(scenario_vs_baseline_df[show_cols2].head(30))

ROOT: /home/tahiti/Malashin_Projects
Output: /home/tahiti/Malashin_Projects/ALL_fractional_experiments_statistics.xlsx
Scanning: /home/tahiti/Malashin_Projects/runs_transformer_fractional
Scanning: /home/tahiti/Malashin_Projects/runs_fractional_scenarios
Scanning: /home/tahiti/Malashin_Projects/runs_fractional_hypothesis_v2
Scanning: /home/tahiti/Malashin_Projects/runs_fractional_focused_final

Collected:
all_runs_df: (3375, 68)
epochs_df  : (33750, 34)
files_df   : (30375, 9)

DONE
CSV: /home/tahiti/Malashin_Projects/ALL_fractional_experiments_all_runs.csv
Excel: /home/tahiti/Malashin_Projects/ALL_fractional_experiments_statistics.xlsx

Overview total:
                      metric  value
0           total_runs_found   3375
1              complete_runs   3375
2  incomplete_or_broken_runs      0
3      total_epochs_recorded  33750

Top 20:
                  experiment_name  dataset_name    model_name  \
0     runs_transformer_fractional  small_D2_L64   T_wide_mean   
1     runs_transfor

In [5]:
from __future__ import annotations

import math
from pathlib import Path
from typing import Optional, List

import pandas as pd


# ============================================================
# CONFIG
# ============================================================

ROOT = Path.cwd()

INPUT_CSV = ROOT / "ALL_fractional_experiments_all_runs.csv"
INPUT_XLSX = ROOT / "ALL_fractional_experiments_statistics.xlsx"

OUT_PREFIX = ROOT / "FINAL_ANALYSIS"

OUT_TXT = ROOT / "FINAL_ANALYSIS_report.txt"
OUT_XLSX = ROOT / "FINAL_ANALYSIS_compact.xlsx"

OUT_SUMMARY_CSV = ROOT / "FINAL_ANALYSIS_summary.csv"
OUT_BY_SCENARIO_CSV = ROOT / "FINAL_ANALYSIS_by_scenario.csv"
OUT_VS_BASELINE_CSV = ROOT / "FINAL_ANALYSIS_vs_baseline.csv"
OUT_TOP_RUNS_CSV = ROOT / "FINAL_ANALYSIS_top_runs.csv"
OUT_BEST_DATASET_CSV = ROOT / "FINAL_ANALYSIS_best_by_dataset.csv"
OUT_BEST_DATASET_MODEL_CSV = ROOT / "FINAL_ANALYSIS_best_by_dataset_model.csv"
OUT_BEST_EXPERIMENT_CSV = ROOT / "FINAL_ANALYSIS_best_by_experiment.csv"
OUT_PROGRESS_CSV = ROOT / "FINAL_ANALYSIS_progress.csv"

BASELINE_NAME = "baseline_adamw"
METRIC = "best_val_macro_f1"

print("ROOT:", ROOT)


# ============================================================
# LOAD
# ============================================================

def load_results() -> pd.DataFrame:
    if INPUT_CSV.exists():
        print("Reading CSV:", INPUT_CSV)
        return pd.read_csv(INPUT_CSV)

    if INPUT_XLSX.exists():
        print("CSV not found. Reading XLSX:", INPUT_XLSX)
        return pd.read_excel(INPUT_XLSX, sheet_name="all_runs")

    raise FileNotFoundError(
        f"Could not find either:\n{INPUT_CSV}\nor\n{INPUT_XLSX}"
    )


df = load_results()

print("Raw shape:", df.shape)
print("Columns:", len(df.columns))


# ============================================================
# CLEANUP
# ============================================================

def to_num(col: str) -> None:
    if col in df.columns:
        df[col] = pd.to_numeric(df[col], errors="coerce")


num_cols = [
    "run_id",
    "n_samples",
    "seq_len",
    "vocab_size",
    "num_classes",
    "mean_length",
    "max_length",
    "d_model",
    "nhead",
    "num_layers",
    "dim_feedforward",
    "dropout",
    "alpha",
    "beta",
    "mix_lambda",
    "start_epoch",
    "lr",
    "weight_decay",
    "n_epochs_recorded",
    "best_epoch",
    "best_val_macro_f1",
    "best_val_accuracy",
    "best_val_loss",
    "best_val_micro_f1",
    "best_val_weighted_f1",
    "final_val_macro_f1",
    "final_val_accuracy",
    "final_val_loss",
    "final_train_macro_f1",
    "final_train_accuracy",
    "final_train_loss",
    "mean_grad_norm",
    "mean_fractional_applied_params",
    "mean_epoch_time",
    "total_epoch_time",
    "mean_throughput",
]

for c in num_cols:
    to_num(c)

for c in [
    "experiment_name",
    "status",
    "dataset_name",
    "model_name",
    "scenario_name",
    "optimizer_name",
    "mode",
    "target",
    "pooling",
]:
    if c in df.columns:
        df[c] = df[c].astype("string").fillna("")


if "status" not in df.columns:
    df["status"] = "unknown"

if "scenario_name" not in df.columns:
    if "optimizer_name" in df.columns:
        df["scenario_name"] = df["optimizer_name"]
    else:
        df["scenario_name"] = "unknown"

if "experiment_name" not in df.columns:
    df["experiment_name"] = "unknown_experiment"

if "is_complete" not in df.columns:
    df["is_complete"] = df["status"].eq("complete")

complete_df = df[df["status"].eq("complete")].copy()

if complete_df.empty:
    raise RuntimeError("No complete runs found. Check input files.")


# ============================================================
# HELPER AGG
# ============================================================

def agg_table(group_cols: List[str], data: Optional[pd.DataFrame] = None) -> pd.DataFrame:
    if data is None:
        data = complete_df

    metrics = [
        "best_val_macro_f1",
        "best_val_accuracy",
        "best_val_loss",
        "final_val_macro_f1",
        "final_val_accuracy",
        "final_val_loss",
        "mean_epoch_time",
        "mean_throughput",
    ]

    metrics = [m for m in metrics if m in data.columns]

    out = (
        data
        .groupby(group_cols, dropna=False)[metrics]
        .agg(["count", "mean", "std", "min", "max"])
        .reset_index()
    )

    out.columns = [
        "_".join([str(x) for x in col if str(x) != ""])
        if isinstance(col, tuple)
        else str(col)
        for col in out.columns
    ]

    if "best_val_macro_f1_mean" in out.columns:
        out = out.sort_values("best_val_macro_f1_mean", ascending=False)

    return out.reset_index(drop=True)


def first_existing(cols: List[str]) -> List[str]:
    return [c for c in cols if c in df.columns]


# ============================================================
# OVERVIEW
# ============================================================

total_runs = len(df)
complete_runs = int(df["status"].eq("complete").sum())
broken_runs = int(~df["status"].eq("complete").sum())
total_experiments = df["experiment_name"].nunique()
total_scenarios = df["scenario_name"].nunique()
total_datasets = df["dataset_name"].nunique() if "dataset_name" in df.columns else None
total_models = df["model_name"].nunique() if "model_name" in df.columns else None

overview_rows = [
    {"metric": "total_rows", "value": total_runs},
    {"metric": "complete_runs", "value": complete_runs},
    {"metric": "non_complete_runs", "value": broken_runs},
    {"metric": "completion_percent_found_rows", "value": 100.0 * complete_runs / max(total_runs, 1)},
    {"metric": "experiments", "value": total_experiments},
    {"metric": "datasets", "value": total_datasets},
    {"metric": "models", "value": total_models},
    {"metric": "scenarios", "value": total_scenarios},
    {"metric": "best_overall_macro_f1", "value": complete_df["best_val_macro_f1"].max()},
    {"metric": "mean_overall_macro_f1", "value": complete_df["best_val_macro_f1"].mean()},
]

summary_df = pd.DataFrame(overview_rows)

progress_df = (
    df
    .groupby(["experiment_name", "status"], dropna=False)
    .size()
    .reset_index(name="count")
    .sort_values(["experiment_name", "status"])
)


# ============================================================
# LEADERBOARD
# ============================================================

leaderboard_cols = first_existing([
    "experiment_name",
    "dataset_name",
    "model_name",
    "scenario_name",
    "run_id",
    "best_val_macro_f1",
    "best_val_accuracy",
    "best_val_loss",
    "best_epoch",
    "mode",
    "target",
    "alpha",
    "beta",
    "mix_lambda",
    "start_epoch",
    "mean_epoch_time",
    "mean_throughput",
    "exp_dir",
])

top_runs_df = (
    complete_df
    .sort_values("best_val_macro_f1", ascending=False)
    [leaderboard_cols]
    .reset_index(drop=True)
)


# ============================================================
# MAIN AGGREGATIONS
# ============================================================

by_experiment_df = agg_table(["experiment_name"])
by_scenario_df = agg_table([
    "experiment_name",
    "scenario_name",
    "mode",
    "target",
    "alpha",
    "beta",
    "mix_lambda",
    "start_epoch",
])

by_scenario_global_df = agg_table([
    "scenario_name",
    "mode",
    "target",
    "alpha",
    "beta",
    "mix_lambda",
    "start_epoch",
])

by_dataset_scenario_df = agg_table([
    "experiment_name",
    "dataset_name",
    "scenario_name",
    "mode",
    "target",
    "alpha",
    "beta",
    "mix_lambda",
    "start_epoch",
])

by_model_scenario_df = agg_table([
    "experiment_name",
    "model_name",
    "scenario_name",
    "mode",
    "target",
    "alpha",
    "beta",
    "mix_lambda",
    "start_epoch",
])

by_dataset_model_scenario_df = agg_table([
    "experiment_name",
    "dataset_name",
    "model_name",
    "scenario_name",
    "mode",
    "target",
    "alpha",
    "beta",
    "mix_lambda",
    "start_epoch",
])


# ============================================================
# BASELINE COMPARISON
# ============================================================

baseline_keys = ["experiment_name", "dataset_name", "model_name", "run_id"]

baseline_df = complete_df[complete_df["scenario_name"].eq(BASELINE_NAME)].copy()

baseline_ref_cols = baseline_keys + [
    "best_val_macro_f1",
    "best_val_accuracy",
    "best_val_loss",
    "final_val_macro_f1",
    "final_val_accuracy",
    "final_val_loss",
]

baseline_ref_cols = [c for c in baseline_ref_cols if c in baseline_df.columns]

baseline_ref = baseline_df[baseline_ref_cols].rename(
    columns={
        "best_val_macro_f1": "baseline_best_val_macro_f1",
        "best_val_accuracy": "baseline_best_val_accuracy",
        "best_val_loss": "baseline_best_val_loss",
        "final_val_macro_f1": "baseline_final_val_macro_f1",
        "final_val_accuracy": "baseline_final_val_accuracy",
        "final_val_loss": "baseline_final_val_loss",
    }
)

comparison_df = complete_df.merge(
    baseline_ref,
    on=baseline_keys,
    how="left",
)

comparison_df["delta_best_val_macro_f1_vs_baseline"] = (
    comparison_df["best_val_macro_f1"] - comparison_df["baseline_best_val_macro_f1"]
)

comparison_df["delta_best_val_accuracy_vs_baseline"] = (
    comparison_df["best_val_accuracy"] - comparison_df["baseline_best_val_accuracy"]
)

comparison_df["delta_final_val_macro_f1_vs_baseline"] = (
    comparison_df["final_val_macro_f1"] - comparison_df["baseline_final_val_macro_f1"]
)

comparison_df["beats_baseline_best_macro_f1"] = (
    comparison_df["delta_best_val_macro_f1_vs_baseline"] > 0
)

comparison_df["has_baseline_pair"] = comparison_df["baseline_best_val_macro_f1"].notna()

comparison_nonbaseline = comparison_df[
    comparison_df["scenario_name"].ne(BASELINE_NAME)
    & comparison_df["has_baseline_pair"]
].copy()

if not comparison_nonbaseline.empty:
    scenario_vs_baseline_df = (
        comparison_nonbaseline
        .groupby([
            "experiment_name",
            "scenario_name",
            "mode",
            "target",
            "alpha",
            "beta",
            "mix_lambda",
            "start_epoch",
        ], dropna=False)[
            [
                "delta_best_val_macro_f1_vs_baseline",
                "delta_best_val_accuracy_vs_baseline",
                "delta_final_val_macro_f1_vs_baseline",
                "beats_baseline_best_macro_f1",
            ]
        ]
        .agg(["count", "mean", "std", "min", "max", "sum"])
        .reset_index()
    )

    scenario_vs_baseline_df.columns = [
        "_".join([str(x) for x in col if str(x) != ""])
        if isinstance(col, tuple)
        else str(col)
        for col in scenario_vs_baseline_df.columns
    ]

    scenario_vs_baseline_df = scenario_vs_baseline_df.sort_values(
        "delta_best_val_macro_f1_vs_baseline_mean",
        ascending=False,
    ).reset_index(drop=True)

    scenario_vs_baseline_global_df = (
        comparison_nonbaseline
        .groupby([
            "scenario_name",
            "mode",
            "target",
            "alpha",
            "beta",
            "mix_lambda",
            "start_epoch",
        ], dropna=False)[
            [
                "delta_best_val_macro_f1_vs_baseline",
                "delta_best_val_accuracy_vs_baseline",
                "delta_final_val_macro_f1_vs_baseline",
                "beats_baseline_best_macro_f1",
            ]
        ]
        .agg(["count", "mean", "std", "min", "max", "sum"])
        .reset_index()
    )

    scenario_vs_baseline_global_df.columns = [
        "_".join([str(x) for x in col if str(x) != ""])
        if isinstance(col, tuple)
        else str(col)
        for col in scenario_vs_baseline_global_df.columns
    ]

    scenario_vs_baseline_global_df = scenario_vs_baseline_global_df.sort_values(
        "delta_best_val_macro_f1_vs_baseline_mean",
        ascending=False,
    ).reset_index(drop=True)
else:
    scenario_vs_baseline_df = pd.DataFrame()
    scenario_vs_baseline_global_df = pd.DataFrame()


# ============================================================
# BEST BY GROUP
# ============================================================

best_by_experiment_df = (
    complete_df
    .sort_values("best_val_macro_f1", ascending=False)
    .groupby(["experiment_name"], as_index=False)
    .head(1)
    .reset_index(drop=True)
)

best_by_dataset_df = (
    complete_df
    .sort_values("best_val_macro_f1", ascending=False)
    .groupby(["experiment_name", "dataset_name"], as_index=False)
    .head(1)
    .reset_index(drop=True)
)

best_by_model_df = (
    complete_df
    .sort_values("best_val_macro_f1", ascending=False)
    .groupby(["experiment_name", "model_name"], as_index=False)
    .head(1)
    .reset_index(drop=True)
)

best_by_dataset_model_df = (
    complete_df
    .sort_values("best_val_macro_f1", ascending=False)
    .groupby(["experiment_name", "dataset_name", "model_name"], as_index=False)
    .head(1)
    .reset_index(drop=True)
)


# ============================================================
# COMPACT FINAL INTERPRETATION TABLE
# ============================================================

def classify_scenario(row) -> str:
    scenario = str(row.get("scenario_name", ""))
    mode = str(row.get("mode", ""))
    target = str(row.get("target", ""))

    if scenario == BASELINE_NAME:
        return "baseline"
    if mode == "replace" and target == "all":
        return "full_replace_negative"
    if target == "embeddings" and mode == "mix":
        return "embedding_mixed_memory"
    if target == "embeddings" and mode == "replace":
        return "embedding_replacement_memory"
    if target == "head" and mode == "mix":
        return "head_mixed_memory"
    if target == "all" and mode == "mix":
        return "all_parameter_weak_mix"
    return "other"


complete_df["scenario_family"] = complete_df.apply(classify_scenario, axis=1)

by_family_df = (
    complete_df
    .groupby(["scenario_family"], dropna=False)[
        [
            "best_val_macro_f1",
            "best_val_accuracy",
            "best_val_loss",
            "mean_epoch_time",
        ]
    ]
    .agg(["count", "mean", "std", "min", "max"])
    .reset_index()
)

by_family_df.columns = [
    "_".join([str(x) for x in col if str(x) != ""])
    if isinstance(col, tuple)
    else str(col)
    for col in by_family_df.columns
]

by_family_df = by_family_df.sort_values(
    "best_val_macro_f1_mean",
    ascending=False,
).reset_index(drop=True)


# ============================================================
# TEXT REPORT
# ============================================================

def fmt(x, nd=4):
    if pd.isna(x):
        return "NA"
    try:
        return f"{float(x):.{nd}f}"
    except Exception:
        return str(x)


lines = []

lines.append("=" * 90)
lines.append("FINAL FRACTIONAL TRANSFORMER EXPERIMENTS REPORT")
lines.append("=" * 90)
lines.append("")
lines.append(f"Input rows: {len(df)}")
lines.append(f"Complete runs: {complete_runs}")
lines.append(f"Non-complete runs: {broken_runs}")
lines.append(f"Experiments: {total_experiments}")
lines.append(f"Datasets: {total_datasets}")
lines.append(f"Models: {total_models}")
lines.append(f"Scenarios: {total_scenarios}")
lines.append("")

best = top_runs_df.iloc[0]
lines.append("BEST SINGLE RUN")
lines.append("-" * 90)
lines.append(
    f"{best.get('experiment_name')} | "
    f"{best.get('dataset_name')} | "
    f"{best.get('model_name')} | "
    f"{best.get('scenario_name')} | "
    f"run={best.get('run_id')} | "
    f"best_macro_f1={fmt(best.get('best_val_macro_f1'))} | "
    f"acc={fmt(best.get('best_val_accuracy'))}"
)
lines.append("")

lines.append("BEST SCENARIOS BY MEAN MACRO F1")
lines.append("-" * 90)
tmp = by_scenario_global_df.head(15)
for _, r in tmp.iterrows():
    lines.append(
        f"{r.get('scenario_name')} | "
        f"family={classify_scenario(r)} | "
        f"mean_f1={fmt(r.get('best_val_macro_f1_mean'))} | "
        f"std={fmt(r.get('best_val_macro_f1_std'))} | "
        f"n={int(r.get('best_val_macro_f1_count', 0))}"
    )
lines.append("")

if not scenario_vs_baseline_global_df.empty:
    lines.append("BEST SCENARIOS VS BASELINE")
    lines.append("-" * 90)
    tmp = scenario_vs_baseline_global_df.head(15)
    for _, r in tmp.iterrows():
        lines.append(
            f"{r.get('scenario_name')} | "
            f"delta_mean={fmt(r.get('delta_best_val_macro_f1_vs_baseline_mean'))} | "
            f"delta_std={fmt(r.get('delta_best_val_macro_f1_vs_baseline_std'))} | "
            f"win_rate={fmt(r.get('beats_baseline_best_macro_f1_mean'))} | "
            f"wins={fmt(r.get('beats_baseline_best_macro_f1_sum'), 0)} / "
            f"{fmt(r.get('delta_best_val_macro_f1_vs_baseline_count'), 0)}"
        )
    lines.append("")

lines.append("BEST BY DATASET")
lines.append("-" * 90)
cols = [
    "experiment_name",
    "dataset_name",
    "model_name",
    "scenario_name",
    "run_id",
    "best_val_macro_f1",
    "best_val_accuracy",
]
for _, r in best_by_dataset_df[cols].iterrows():
    lines.append(
        f"{r.get('experiment_name')} | {r.get('dataset_name')} | "
        f"{r.get('model_name')} | {r.get('scenario_name')} | "
        f"run={r.get('run_id')} | f1={fmt(r.get('best_val_macro_f1'))} | "
        f"acc={fmt(r.get('best_val_accuracy'))}"
    )
lines.append("")

lines.append("SCENARIO FAMILY SUMMARY")
lines.append("-" * 90)
for _, r in by_family_df.iterrows():
    lines.append(
        f"{r.get('scenario_family')} | "
        f"mean_f1={fmt(r.get('best_val_macro_f1_mean'))} | "
        f"std={fmt(r.get('best_val_macro_f1_std'))} | "
        f"max={fmt(r.get('best_val_macro_f1_max'))} | "
        f"n={int(r.get('best_val_macro_f1_count', 0))}"
    )
lines.append("")

lines.append("INTERPRETATION")
lines.append("-" * 90)
lines.append(
    "1. Full replacement of all gradients should be treated as a negative control, "
    "because earlier experiments showed it consistently damages optimization."
)
lines.append(
    "2. The main promising direction is weak/selective fractional memory: "
    "head-mix, embeddings-mix, delayed embeddings-mix, and weak all-parameter mix."
)
lines.append(
    "3. The key claim should be modest and careful: selective fractional memory can "
    "slightly improve or match AdamW in several regimes, not dominate it universally."
)
lines.append(
    "4. The most important tables for paper/report writing are scenario_vs_baseline, "
    "by_dataset_scenario, by_dataset_model_scenario, and leaderboard."
)

report_text = "\n".join(lines)
OUT_TXT.write_text(report_text, encoding="utf-8")

print(report_text)


# ============================================================
# SAVE CSV FILES
# ============================================================

summary_df.to_csv(OUT_SUMMARY_CSV, index=False)
progress_df.to_csv(OUT_PROGRESS_CSV, index=False)
by_scenario_global_df.to_csv(OUT_BY_SCENARIO_CSV, index=False)
scenario_vs_baseline_global_df.to_csv(OUT_VS_BASELINE_CSV, index=False)
top_runs_df.to_csv(OUT_TOP_RUNS_CSV, index=False)
best_by_dataset_df.to_csv(OUT_BEST_DATASET_CSV, index=False)
best_by_dataset_model_df.to_csv(OUT_BEST_DATASET_MODEL_CSV, index=False)
best_by_experiment_df.to_csv(OUT_BEST_EXPERIMENT_CSV, index=False)


# ============================================================
# SAVE XLSX IF POSSIBLE
# ============================================================

sheets = {
    "00_summary": summary_df,
    "01_progress": progress_df,
    "02_top_runs": top_runs_df.head(200),
    "03_by_scenario_global": by_scenario_global_df,
    "04_vs_baseline_global": scenario_vs_baseline_global_df,
    "05_by_family": by_family_df,
    "06_by_experiment": by_experiment_df,
    "07_by_dataset_scenario": by_dataset_scenario_df,
    "08_by_model_scenario": by_model_scenario_df,
    "09_by_dataset_model_scenario": by_dataset_model_scenario_df,
    "10_best_by_experiment": best_by_experiment_df,
    "11_best_by_dataset": best_by_dataset_df,
    "12_best_by_dataset_model": best_by_dataset_model_df,
    "13_broken_runs": df[df["status"].ne("complete")],
}

try:
    with pd.ExcelWriter(OUT_XLSX, engine="openpyxl") as writer:
        for sheet_name, table in sheets.items():
            safe_sheet = sheet_name[:31]
            table.to_excel(writer, sheet_name=safe_sheet, index=False)

    print("\nSaved XLSX:", OUT_XLSX)

except Exception as e:
    print("\nCould not save XLSX. CSV files were saved successfully.")
    print("XLSX error:", repr(e))
    print("Install openpyxl if needed:")
    print("pip install openpyxl")


print("\nSaved files:")
for p in [
    OUT_TXT,
    OUT_SUMMARY_CSV,
    OUT_PROGRESS_CSV,
    OUT_BY_SCENARIO_CSV,
    OUT_VS_BASELINE_CSV,
    OUT_TOP_RUNS_CSV,
    OUT_BEST_DATASET_CSV,
    OUT_BEST_DATASET_MODEL_CSV,
    OUT_BEST_EXPERIMENT_CSV,
]:
    print(" ", p)

if OUT_XLSX.exists():
    print(" ", OUT_XLSX)

ROOT: /home/tahiti/Malashin_Projects
Reading CSV: /home/tahiti/Malashin_Projects/ALL_fractional_experiments_all_runs.csv
Raw shape: (3375, 68)
Columns: 68
FINAL FRACTIONAL TRANSFORMER EXPERIMENTS REPORT

Input rows: 3375
Complete runs: 3375
Non-complete runs: -3376
Experiments: 4
Datasets: 3
Models: 5
Scenarios: 51

BEST SINGLE RUN
------------------------------------------------------------------------------------------
runs_transformer_fractional | small_D2_L64 | T_wide_mean | adamw | run=0 | best_macro_f1=0.3222 | acc=0.3650

BEST SCENARIOS BY MEAN MACRO F1
------------------------------------------------------------------------------------------
adamw | family=other | mean_f1=0.1727 | std=0.0802 | n=45
adam | family=other | mean_f1=0.1707 | std=0.0765 | n=45
delayed_head_mix_a08_lam010_warm3 | family=head_mixed_memory | mean_f1=0.1641 | std=0.0743 | n=45
embeddings_only_replace_a08 | family=embedding_replacement_memory | mean_f1=0.1641 | std=0.0751 | n=45
emb_mix_a08_lam005 | famil

In [6]:
from __future__ import annotations

import json
import re
from pathlib import Path
from typing import Any, Dict, Optional, List

import pandas as pd


# ============================================================
# CONFIG
# ============================================================

ROOT = Path("/home/tahiti/Malashin_Projects")

RUN_DIR = ROOT / "runs_real_text_fractional_concept"

OUT_FULL_XLSX = ROOT / "concept_full_summary.xlsx"
OUT_FULL_CSV = ROOT / "concept_full_summary.csv"

OUT_BRIEF_XLSX = ROOT / "concept_brief_summary.xlsx"
OUT_BRIEF_CSV = ROOT / "concept_brief_summary.csv"
OUT_BRIEF_TXT = ROOT / "concept_brief_report.txt"

EXPECTED_EPOCHS = 5
BASELINE_NAME = "baseline_adamw"

PRIMARY_METRIC = "best_test_macro_f1_at_best_val"
SECONDARY_METRIC = "best_val_macro_f1"

print("ROOT:", ROOT)
print("RUN_DIR:", RUN_DIR)

if not RUN_DIR.exists():
    raise FileNotFoundError(f"Run dir not found: {RUN_DIR}")


# ============================================================
# HELPERS
# ============================================================

def read_json(path: Path) -> Optional[Dict[str, Any]]:
    if not path.exists():
        return None
    try:
        return json.loads(path.read_text(encoding="utf-8"))
    except Exception:
        return None


def read_text(path: Path) -> str:
    if not path.exists():
        return ""
    try:
        return path.read_text(encoding="utf-8", errors="ignore")
    except Exception:
        return ""


def safe_float(x) -> Optional[float]:
    try:
        if x is None:
            return None
        v = float(x)
        if pd.isna(v):
            return None
        return v
    except Exception:
        return None


def parse_log_best(text: str) -> Dict[str, Any]:
    out = {}

    patterns = {
        "log_best_epoch": r"BEST EPOCH\s*:\s*([0-9]+)",
        "log_best_val_macro_f1": r"BEST VAL MACRO F1\s*:\s*([0-9eE\.\+\-]+)",
        "log_best_val_acc": r"BEST VAL ACC\s*:\s*([0-9eE\.\+\-]+)",
        "log_best_val_loss": r"BEST VAL LOSS\s*:\s*([0-9eE\.\+\-]+)",
        "log_test_macro_f1_at_best_val": r"TEST MACRO F1 AT BEST VAL\s*:\s*([0-9eE\.\+\-]+)",
        "log_test_acc_at_best_val": r"TEST ACC AT BEST VAL\s*:\s*([0-9eE\.\+\-]+)",
        "log_test_loss_at_best_val": r"TEST LOSS AT BEST VAL\s*:\s*([0-9eE\.\+\-]+)",
        "log_total_run_time_sec": r"TOTAL RUN TIME\s*:\s*([0-9eE\.\+\-]+)",
    }

    for k, pat in patterns.items():
        m = re.search(pat, text)
        if m:
            try:
                out[k] = int(m.group(1)) if "epoch" in k else float(m.group(1))
            except Exception:
                out[k] = m.group(1)

    return out


def best_index_from_history(history: Optional[Dict[str, Any]], metric: str = "val_macro_f1") -> Optional[int]:
    if not isinstance(history, dict):
        return None

    vals = history.get(metric, [])
    if not isinstance(vals, list) or len(vals) == 0:
        return None

    s = pd.to_numeric(pd.Series(vals), errors="coerce")
    if not s.notna().any():
        return None

    return int(s.idxmax())


def list_get(values, idx):
    if isinstance(values, list) and idx is not None and 0 <= idx < len(values):
        return values[idx]
    return None


def list_last(values):
    if isinstance(values, list) and values:
        return values[-1]
    return None


def list_mean(values):
    if isinstance(values, list) and values:
        s = pd.to_numeric(pd.Series(values), errors="coerce")
        if s.notna().any():
            return float(s.mean())
    return None


def list_sum(values):
    if isinstance(values, list) and values:
        s = pd.to_numeric(pd.Series(values), errors="coerce")
        if s.notna().any():
            return float(s.sum())
    return None


def history_metrics(history: Optional[Dict[str, Any]]) -> Dict[str, Any]:
    if not isinstance(history, dict):
        return {"n_epochs_recorded": 0}

    max_epochs = max(
        [len(v) for v in history.values() if isinstance(v, list)] or [0]
    )

    best_idx = best_index_from_history(history, "val_macro_f1")

    out = {
        "n_epochs_recorded": max_epochs,
        "best_epoch": best_idx + 1 if best_idx is not None else None,

        "best_val_macro_f1": list_get(history.get("val_macro_f1"), best_idx),
        "best_val_accuracy": list_get(history.get("val_accuracy"), best_idx),
        "best_val_loss": list_get(history.get("val_loss"), best_idx),
        "best_val_micro_f1": list_get(history.get("val_micro_f1"), best_idx),
        "best_val_weighted_f1": list_get(history.get("val_weighted_f1"), best_idx),
        "best_val_confidence": list_get(history.get("val_confidence"), best_idx),

        "best_test_macro_f1_at_best_val": list_get(history.get("test_macro_f1"), best_idx),
        "best_test_accuracy_at_best_val": list_get(history.get("test_accuracy"), best_idx),
        "best_test_loss_at_best_val": list_get(history.get("test_loss"), best_idx),
        "best_test_micro_f1_at_best_val": list_get(history.get("test_micro_f1"), best_idx),
        "best_test_weighted_f1_at_best_val": list_get(history.get("test_weighted_f1"), best_idx),

        "final_train_loss": list_last(history.get("train_loss")),
        "final_train_accuracy": list_last(history.get("train_accuracy")),
        "final_train_macro_f1": list_last(history.get("train_macro_f1")),

        "final_val_loss": list_last(history.get("val_loss")),
        "final_val_accuracy": list_last(history.get("val_accuracy")),
        "final_val_macro_f1": list_last(history.get("val_macro_f1")),
        "final_val_micro_f1": list_last(history.get("val_micro_f1")),
        "final_val_weighted_f1": list_last(history.get("val_weighted_f1")),

        "final_test_loss": list_last(history.get("test_loss")),
        "final_test_accuracy": list_last(history.get("test_accuracy")),
        "final_test_macro_f1": list_last(history.get("test_macro_f1")),
        "final_test_micro_f1": list_last(history.get("test_micro_f1")),
        "final_test_weighted_f1": list_last(history.get("test_weighted_f1")),

        "mean_grad_norm": list_mean(history.get("grad_norm")),
        "mean_nonfinite_grad_tensors": list_mean(history.get("nonfinite_grad_tensors")),
        "total_nonfinite_train_loss_batches": list_sum(history.get("nonfinite_train_loss_batches")),
        "total_nonfinite_val_loss_batches": list_sum(history.get("nonfinite_val_loss_batches")),
        "total_nonfinite_test_loss_batches": list_sum(history.get("nonfinite_test_loss_batches")),
        "mean_fractional_applied_params": list_mean(history.get("fractional_applied_params")),
        "mean_epoch_time": list_mean(history.get("epoch_time")),
        "total_epoch_time": list_sum(history.get("epoch_time")),
        "mean_throughput": list_mean(history.get("throughput_samples_per_sec")),
    }

    return out


def flatten_columns(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df.columns = [
        "_".join([str(x) for x in col if str(x) != ""])
        if isinstance(col, tuple)
        else str(col)
        for col in df.columns
    ]
    return df


def classify_family(row) -> str:
    scenario = str(row.get("scenario_name", ""))
    mode = str(row.get("mode", ""))
    target = str(row.get("target", ""))

    if scenario == BASELINE_NAME:
        return "baseline"

    if scenario.startswith("negative_full_replace"):
        return "negative_global_replace"

    if scenario.startswith("emb_replace"):
        return "embedding_alpha_sweep_replace"

    if scenario.startswith("emb_mix"):
        return "embedding_weak_mix"

    if scenario.startswith("bottom1"):
        return "bottom1_layer_replace"

    if scenario.startswith("bottom2"):
        return "bottom2_layer_replace"

    if scenario.startswith("emb_plus_bottom1"):
        return "embedding_plus_bottom1_replace"

    if scenario.startswith("delayed_head"):
        return "head_mixed_control"

    if mode == "replace" and target == "all":
        return "negative_global_replace"

    return "other"


# ============================================================
# COLLECT RUNS
# ============================================================

run_rows = []
epoch_rows = []
file_rows = []

run_dirs = sorted(RUN_DIR.glob("*/*/*/run_*"))

print("Found run dirs:", len(run_dirs))

for exp_dir in run_dirs:
    if not exp_dir.is_dir():
        continue

    rel = exp_dir.relative_to(RUN_DIR).parts
    if len(rel) < 4:
        continue

    dataset_name_from_path = rel[0]
    model_name_from_path = rel[1]
    scenario_name_from_path = rel[2]
    run_name = rel[3]

    try:
        run_id = int(run_name.replace("run_", ""))
    except Exception:
        run_id = None

    history_path = exp_dir / "history.json"
    log_path = exp_dir / "log.txt"
    dataset_config_path = exp_dir / "dataset_config.json"
    model_config_path = exp_dir / "model_config.json"
    scenario_config_path = exp_dir / "scenario_config.json"

    initial_model_path = exp_dir / "initial_model.pt"
    best_model_path = exp_dir / "best_model.pt"
    final_model_path = exp_dir / "final_model.pt"

    history = read_json(history_path)
    dataset_cfg = read_json(dataset_config_path) or {}
    model_cfg = read_json(model_config_path) or {}
    scenario_cfg = read_json(scenario_config_path) or {}

    log_text = read_text(log_path)

    hm = history_metrics(history)
    lm = parse_log_best(log_text)

    n_epochs_recorded = hm.get("n_epochs_recorded", 0)

    complete = (
        history_path.exists()
        and best_model_path.exists()
        and n_epochs_recorded >= EXPECTED_EPOCHS
    )

    if complete:
        status = "complete"
    elif history_path.exists() or log_path.exists():
        status = "incomplete_or_broken"
    else:
        status = "empty_or_missing"

    row = {
        "status": status,
        "is_complete": complete,

        "dataset_name": dataset_cfg.get("dataset_name", dataset_name_from_path),
        "task_type": dataset_cfg.get("task_type"),
        "num_labels": dataset_cfg.get("num_labels"),

        "model_name": model_cfg.get("model_name", model_name_from_path),
        "model_family": model_cfg.get("family"),
        "model_loader": model_cfg.get("loader"),
        "model_path": model_cfg.get("model_path"),

        "scenario_name": scenario_cfg.get("scenario_name", scenario_name_from_path),
        "base_optimizer": scenario_cfg.get("base_optimizer"),
        "lr": scenario_cfg.get("lr"),
        "weight_decay": scenario_cfg.get("weight_decay"),
        "mode": scenario_cfg.get("mode"),
        "target": scenario_cfg.get("target"),
        "alpha": scenario_cfg.get("alpha"),
        "beta": scenario_cfg.get("beta"),
        "mix_lambda": scenario_cfg.get("mix_lambda"),
        "start_epoch": scenario_cfg.get("start_epoch"),
        "trainable_scope": scenario_cfg.get("trainable_scope"),

        "run_id": run_id,

        "exp_dir": str(exp_dir),
        "history_path": str(history_path),
        "log_path": str(log_path),
        "dataset_config_path": str(dataset_config_path),
        "model_config_path": str(model_config_path),
        "scenario_config_path": str(scenario_config_path),
        "initial_model_path": str(initial_model_path),
        "best_model_path": str(best_model_path),
        "final_model_path": str(final_model_path),

        "has_history_json": history_path.exists(),
        "has_log_txt": log_path.exists(),
        "has_initial_model_pt": initial_model_path.exists(),
        "has_best_model_pt": best_model_path.exists(),
        "has_final_model_pt": final_model_path.exists(),

        "best_model_size_mb": best_model_path.stat().st_size / 1024**2 if best_model_path.exists() else None,
        "final_model_size_mb": final_model_path.stat().st_size / 1024**2 if final_model_path.exists() else None,
    }

    row.update(hm)
    row.update(lm)

    row["scenario_family"] = classify_family(row)

    run_rows.append(row)

    if isinstance(history, dict):
        max_epochs = max([len(v) for v in history.values() if isinstance(v, list)] or [0])

        for eidx in range(max_epochs):
            erow = {
                "dataset_name": row["dataset_name"],
                "model_name": row["model_name"],
                "scenario_name": row["scenario_name"],
                "scenario_family": row["scenario_family"],
                "run_id": run_id,
                "epoch": eidx + 1,

                "mode": row["mode"],
                "target": row["target"],
                "alpha": row["alpha"],
                "beta": row["beta"],
                "mix_lambda": row["mix_lambda"],
                "start_epoch": row["start_epoch"],
                "trainable_scope": row["trainable_scope"],
            }

            for key, vals in history.items():
                if isinstance(vals, list):
                    erow[key] = vals[eidx] if eidx < len(vals) else None

            epoch_rows.append(erow)

    for f in [
        history_path,
        log_path,
        dataset_config_path,
        model_config_path,
        scenario_config_path,
        initial_model_path,
        best_model_path,
        final_model_path,
    ]:
        file_rows.append(
            {
                "dataset_name": row["dataset_name"],
                "model_name": row["model_name"],
                "scenario_name": row["scenario_name"],
                "run_id": run_id,
                "file_name": f.name,
                "file_path": str(f),
                "exists": f.exists(),
                "size_mb": f.stat().st_size / 1024**2 if f.exists() else None,
            }
        )


all_runs_df = pd.DataFrame(run_rows)
epochs_df = pd.DataFrame(epoch_rows)
files_df = pd.DataFrame(file_rows)

print("all_runs_df:", all_runs_df.shape)
print("epochs_df:", epochs_df.shape)
print("files_df:", files_df.shape)

if all_runs_df.empty:
    raise RuntimeError("No run data found.")


# ============================================================
# NUMERIC CLEANUP
# ============================================================

numeric_candidates = [
    "num_labels",
    "lr",
    "weight_decay",
    "alpha",
    "beta",
    "mix_lambda",
    "start_epoch",
    "run_id",
    "n_epochs_recorded",
    "best_epoch",

    "best_val_macro_f1",
    "best_val_accuracy",
    "best_val_loss",
    "best_val_micro_f1",
    "best_val_weighted_f1",
    "best_test_macro_f1_at_best_val",
    "best_test_accuracy_at_best_val",
    "best_test_loss_at_best_val",
    "best_test_micro_f1_at_best_val",
    "best_test_weighted_f1_at_best_val",

    "final_train_loss",
    "final_train_accuracy",
    "final_train_macro_f1",
    "final_val_loss",
    "final_val_accuracy",
    "final_val_macro_f1",
    "final_test_loss",
    "final_test_accuracy",
    "final_test_macro_f1",

    "mean_grad_norm",
    "mean_nonfinite_grad_tensors",
    "total_nonfinite_train_loss_batches",
    "total_nonfinite_val_loss_batches",
    "total_nonfinite_test_loss_batches",
    "mean_fractional_applied_params",
    "mean_epoch_time",
    "total_epoch_time",
    "mean_throughput",
    "best_model_size_mb",
    "final_model_size_mb",
]

for c in numeric_candidates:
    if c in all_runs_df.columns:
        all_runs_df[c] = pd.to_numeric(all_runs_df[c], errors="coerce")

for c in epochs_df.columns:
    if c not in {
        "dataset_name",
        "model_name",
        "scenario_name",
        "scenario_family",
        "mode",
        "target",
        "trainable_scope",
    }:
        epochs_df[c] = pd.to_numeric(epochs_df[c], errors="coerce")


complete_df = all_runs_df[all_runs_df["status"].eq("complete")].copy()
broken_df = all_runs_df[~all_runs_df["status"].eq("complete")].copy()

print("complete:", complete_df.shape)
print("broken:", broken_df.shape)


# ============================================================
# OVERVIEW
# ============================================================

overview_df = pd.DataFrame([
    {"metric": "run_dirs_found", "value": len(run_dirs)},
    {"metric": "rows_collected", "value": len(all_runs_df)},
    {"metric": "complete_runs", "value": int((all_runs_df["status"] == "complete").sum())},
    {"metric": "non_complete_runs", "value": int((all_runs_df["status"] != "complete").sum())},
    {"metric": "datasets", "value": all_runs_df["dataset_name"].nunique()},
    {"metric": "models", "value": all_runs_df["model_name"].nunique()},
    {"metric": "scenarios", "value": all_runs_df["scenario_name"].nunique()},
    {"metric": "runs_per_config_max", "value": all_runs_df["run_id"].nunique()},
    {"metric": "expected_epochs", "value": EXPECTED_EPOCHS},
    {"metric": "best_single_test_macro_f1_at_best_val", "value": complete_df[PRIMARY_METRIC].max() if not complete_df.empty else None},
    {"metric": "mean_test_macro_f1_at_best_val", "value": complete_df[PRIMARY_METRIC].mean() if not complete_df.empty else None},
])


progress_df = (
    all_runs_df
    .groupby(["dataset_name", "model_name", "scenario_name", "status"], dropna=False)
    .size()
    .reset_index(name="count")
    .sort_values(["dataset_name", "model_name", "scenario_name", "status"])
)


# ============================================================
# AGGREGATIONS
# ============================================================

metrics = [
    "best_test_macro_f1_at_best_val",
    "best_test_accuracy_at_best_val",
    "best_test_loss_at_best_val",
    "best_val_macro_f1",
    "best_val_accuracy",
    "best_val_loss",
    "final_test_macro_f1",
    "final_test_accuracy",
    "final_val_macro_f1",
    "mean_grad_norm",
    "mean_nonfinite_grad_tensors",
    "total_nonfinite_train_loss_batches",
    "mean_fractional_applied_params",
    "mean_epoch_time",
    "mean_throughput",
    "total_epoch_time",
]

metrics = [m for m in metrics if m in complete_df.columns]


def agg_by(cols: List[str], data: Optional[pd.DataFrame] = None) -> pd.DataFrame:
    if data is None:
        data = complete_df

    if data.empty:
        return pd.DataFrame()

    out = (
        data
        .groupby(cols, dropna=False)[metrics]
        .agg(["count", "mean", "std", "min", "max"])
        .reset_index()
    )

    out = flatten_columns(out)

    sort_col = f"{PRIMARY_METRIC}_mean"
    if sort_col in out.columns:
        out = out.sort_values(sort_col, ascending=False)

    return out.reset_index(drop=True)


by_scenario_global_df = agg_by([
    "scenario_name",
    "scenario_family",
    "mode",
    "target",
    "alpha",
    "beta",
    "mix_lambda",
    "start_epoch",
    "trainable_scope",
])

by_dataset_scenario_df = agg_by([
    "dataset_name",
    "scenario_name",
    "scenario_family",
    "mode",
    "target",
    "alpha",
    "beta",
    "mix_lambda",
    "start_epoch",
    "trainable_scope",
])

by_model_scenario_df = agg_by([
    "model_name",
    "scenario_name",
    "scenario_family",
    "mode",
    "target",
    "alpha",
    "beta",
    "mix_lambda",
    "start_epoch",
    "trainable_scope",
])

by_dataset_model_scenario_df = agg_by([
    "dataset_name",
    "model_name",
    "scenario_name",
    "scenario_family",
    "mode",
    "target",
    "alpha",
    "beta",
    "mix_lambda",
    "start_epoch",
    "trainable_scope",
])

by_family_df = agg_by([
    "scenario_family",
])


leaderboard_df = (
    complete_df
    .sort_values(PRIMARY_METRIC, ascending=False)
    .reset_index(drop=True)
)


# ============================================================
# BASELINE COMPARISON
# ============================================================

baseline_keys = ["dataset_name", "model_name", "run_id"]

baseline_df = complete_df[complete_df["scenario_name"].eq(BASELINE_NAME)].copy()

baseline_cols = baseline_keys + [
    "best_test_macro_f1_at_best_val",
    "best_test_accuracy_at_best_val",
    "best_val_macro_f1",
    "best_val_accuracy",
    "final_test_macro_f1",
    "final_val_macro_f1",
]

baseline_cols = [c for c in baseline_cols if c in baseline_df.columns]

baseline_ref = baseline_df[baseline_cols].rename(
    columns={
        "best_test_macro_f1_at_best_val": "baseline_best_test_macro_f1_at_best_val",
        "best_test_accuracy_at_best_val": "baseline_best_test_accuracy_at_best_val",
        "best_val_macro_f1": "baseline_best_val_macro_f1",
        "best_val_accuracy": "baseline_best_val_accuracy",
        "final_test_macro_f1": "baseline_final_test_macro_f1",
        "final_val_macro_f1": "baseline_final_val_macro_f1",
    }
)

comparison_df = complete_df.merge(
    baseline_ref,
    on=baseline_keys,
    how="left",
)

comparison_df["delta_test_macro_f1_vs_baseline"] = (
    comparison_df["best_test_macro_f1_at_best_val"]
    - comparison_df["baseline_best_test_macro_f1_at_best_val"]
)

comparison_df["delta_test_accuracy_vs_baseline"] = (
    comparison_df["best_test_accuracy_at_best_val"]
    - comparison_df["baseline_best_test_accuracy_at_best_val"]
)

comparison_df["delta_val_macro_f1_vs_baseline"] = (
    comparison_df["best_val_macro_f1"]
    - comparison_df["baseline_best_val_macro_f1"]
)

comparison_df["beats_baseline_test_macro_f1"] = (
    comparison_df["delta_test_macro_f1_vs_baseline"] > 0
)

comparison_df["has_baseline_pair"] = comparison_df["baseline_best_test_macro_f1_at_best_val"].notna()

comparison_nonbaseline_df = comparison_df[
    comparison_df["scenario_name"].ne(BASELINE_NAME)
    & comparison_df["has_baseline_pair"]
].copy()

if not comparison_nonbaseline_df.empty:
    vs_baseline_global_df = (
        comparison_nonbaseline_df
        .groupby([
            "scenario_name",
            "scenario_family",
            "mode",
            "target",
            "alpha",
            "beta",
            "mix_lambda",
            "start_epoch",
            "trainable_scope",
        ], dropna=False)[
            [
                "delta_test_macro_f1_vs_baseline",
                "delta_test_accuracy_vs_baseline",
                "delta_val_macro_f1_vs_baseline",
                "beats_baseline_test_macro_f1",
            ]
        ]
        .agg(["count", "mean", "std", "min", "max", "sum"])
        .reset_index()
    )
    vs_baseline_global_df = flatten_columns(vs_baseline_global_df)

    if "delta_test_macro_f1_vs_baseline_mean" in vs_baseline_global_df.columns:
        vs_baseline_global_df = vs_baseline_global_df.sort_values(
            "delta_test_macro_f1_vs_baseline_mean",
            ascending=False
        ).reset_index(drop=True)

    vs_baseline_by_dataset_df = (
        comparison_nonbaseline_df
        .groupby([
            "dataset_name",
            "scenario_name",
            "scenario_family",
            "mode",
            "target",
            "alpha",
            "beta",
            "mix_lambda",
            "start_epoch",
            "trainable_scope",
        ], dropna=False)[
            [
                "delta_test_macro_f1_vs_baseline",
                "delta_test_accuracy_vs_baseline",
                "delta_val_macro_f1_vs_baseline",
                "beats_baseline_test_macro_f1",
            ]
        ]
        .agg(["count", "mean", "std", "min", "max", "sum"])
        .reset_index()
    )
    vs_baseline_by_dataset_df = flatten_columns(vs_baseline_by_dataset_df)

    if "delta_test_macro_f1_vs_baseline_mean" in vs_baseline_by_dataset_df.columns:
        vs_baseline_by_dataset_df = vs_baseline_by_dataset_df.sort_values(
            ["dataset_name", "delta_test_macro_f1_vs_baseline_mean"],
            ascending=[True, False]
        ).reset_index(drop=True)

    vs_baseline_by_dataset_model_df = (
        comparison_nonbaseline_df
        .groupby([
            "dataset_name",
            "model_name",
            "scenario_name",
            "scenario_family",
            "mode",
            "target",
            "alpha",
            "beta",
            "mix_lambda",
            "start_epoch",
            "trainable_scope",
        ], dropna=False)[
            [
                "delta_test_macro_f1_vs_baseline",
                "delta_test_accuracy_vs_baseline",
                "delta_val_macro_f1_vs_baseline",
                "beats_baseline_test_macro_f1",
            ]
        ]
        .agg(["count", "mean", "std", "min", "max", "sum"])
        .reset_index()
    )
    vs_baseline_by_dataset_model_df = flatten_columns(vs_baseline_by_dataset_model_df)

else:
    vs_baseline_global_df = pd.DataFrame()
    vs_baseline_by_dataset_df = pd.DataFrame()
    vs_baseline_by_dataset_model_df = pd.DataFrame()


# ============================================================
# BEST TABLES
# ============================================================

best_by_dataset_df = (
    complete_df
    .sort_values(PRIMARY_METRIC, ascending=False)
    .groupby(["dataset_name"], as_index=False)
    .head(1)
    .reset_index(drop=True)
)

best_by_model_df = (
    complete_df
    .sort_values(PRIMARY_METRIC, ascending=False)
    .groupby(["model_name"], as_index=False)
    .head(1)
    .reset_index(drop=True)
)

best_by_dataset_model_df = (
    complete_df
    .sort_values(PRIMARY_METRIC, ascending=False)
    .groupby(["dataset_name", "model_name"], as_index=False)
    .head(1)
    .reset_index(drop=True)
)

best_by_scenario_df = (
    complete_df
    .sort_values(PRIMARY_METRIC, ascending=False)
    .groupby(["scenario_name"], as_index=False)
    .head(1)
    .reset_index(drop=True)
)


# ============================================================
# BRIEF TABLE
# ============================================================

brief_cols = [
    "scenario_name",
    "scenario_family",
    "mode",
    "target",
    "alpha",
    "beta",
    "mix_lambda",
    "start_epoch",
    "trainable_scope",
    f"{PRIMARY_METRIC}_count",
    f"{PRIMARY_METRIC}_mean",
    f"{PRIMARY_METRIC}_std",
    f"{PRIMARY_METRIC}_min",
    f"{PRIMARY_METRIC}_max",
    "best_val_macro_f1_mean",
    "final_test_macro_f1_mean",
    "mean_nonfinite_grad_tensors_mean",
    "total_nonfinite_train_loss_batches_mean",
    "mean_fractional_applied_params_mean",
]

brief_cols = [c for c in brief_cols if c in by_scenario_global_df.columns]

brief_summary_df = by_scenario_global_df[brief_cols].copy()

if not vs_baseline_global_df.empty:
    add_cols = [
        "scenario_name",
        "delta_test_macro_f1_vs_baseline_count",
        "delta_test_macro_f1_vs_baseline_mean",
        "delta_test_macro_f1_vs_baseline_std",
        "delta_test_macro_f1_vs_baseline_min",
        "delta_test_macro_f1_vs_baseline_max",
        "beats_baseline_test_macro_f1_mean",
        "beats_baseline_test_macro_f1_sum",
    ]
    add_cols = [c for c in add_cols if c in vs_baseline_global_df.columns]

    brief_summary_df = brief_summary_df.merge(
        vs_baseline_global_df[add_cols],
        on="scenario_name",
        how="left",
    )

brief_summary_df = brief_summary_df.sort_values(
    f"{PRIMARY_METRIC}_mean",
    ascending=False,
).reset_index(drop=True)


# ============================================================
# TEXT REPORT
# ============================================================

def fmt(x, nd=4):
    try:
        if pd.isna(x):
            return "NA"
        return f"{float(x):.{nd}f}"
    except Exception:
        return str(x)


lines = []
lines.append("=" * 100)
lines.append("REAL TEXT FRACTIONAL CONCEPT SUMMARY")
lines.append("=" * 100)
lines.append("")
lines.append(f"Run dir: {RUN_DIR}")
lines.append(f"Expected epochs: {EXPECTED_EPOCHS}")
lines.append("")
lines.append(f"Found rows: {len(all_runs_df)}")
lines.append(f"Complete runs: {len(complete_df)}")
lines.append(f"Non-complete runs: {len(broken_df)}")
lines.append(f"Datasets: {all_runs_df['dataset_name'].nunique()}")
lines.append(f"Models: {all_runs_df['model_name'].nunique()}")
lines.append(f"Scenarios: {all_runs_df['scenario_name'].nunique()}")
lines.append("")

if not leaderboard_df.empty:
    top = leaderboard_df.iloc[0]
    lines.append("BEST SINGLE RUN")
    lines.append("-" * 100)
    lines.append(
        f"{top.get('dataset_name')} | {top.get('model_name')} | "
        f"{top.get('scenario_name')} | run={top.get('run_id')} | "
        f"test_macro_f1@best_val={fmt(top.get(PRIMARY_METRIC))} | "
        f"val_macro_f1={fmt(top.get(SECONDARY_METRIC))}"
    )
    lines.append("")

lines.append("TOP SCENARIOS BY MEAN TEST MACRO-F1 @ BEST VAL")
lines.append("-" * 100)

for _, r in brief_summary_df.head(15).iterrows():
    lines.append(
        f"{r.get('scenario_name')} | "
        f"family={r.get('scenario_family')} | "
        f"mean={fmt(r.get(f'{PRIMARY_METRIC}_mean'))} | "
        f"std={fmt(r.get(f'{PRIMARY_METRIC}_std'))} | "
        f"n={fmt(r.get(f'{PRIMARY_METRIC}_count'), 0)}"
    )

lines.append("")

if not vs_baseline_global_df.empty:
    lines.append("TOP SCENARIOS VS BASELINE")
    lines.append("-" * 100)

    for _, r in vs_baseline_global_df.head(15).iterrows():
        lines.append(
            f"{r.get('scenario_name')} | "
            f"delta_mean={fmt(r.get('delta_test_macro_f1_vs_baseline_mean'))} | "
            f"delta_std={fmt(r.get('delta_test_macro_f1_vs_baseline_std'))} | "
            f"win_rate={fmt(r.get('beats_baseline_test_macro_f1_mean'))} | "
            f"wins={fmt(r.get('beats_baseline_test_macro_f1_sum'), 0)} / "
            f"{fmt(r.get('delta_test_macro_f1_vs_baseline_count'), 0)}"
        )

lines.append("")
lines.append("BEST BY DATASET")
lines.append("-" * 100)

for _, r in best_by_dataset_df.iterrows():
    lines.append(
        f"{r.get('dataset_name')} | {r.get('model_name')} | "
        f"{r.get('scenario_name')} | run={r.get('run_id')} | "
        f"test_macro_f1@best_val={fmt(r.get(PRIMARY_METRIC))}"
    )

lines.append("")
lines.append("FILES CREATED")
lines.append("-" * 100)
lines.append(str(OUT_FULL_XLSX))
lines.append(str(OUT_FULL_CSV))
lines.append(str(OUT_BRIEF_XLSX))
lines.append(str(OUT_BRIEF_CSV))
lines.append(str(OUT_BRIEF_TXT))

report_text = "\n".join(lines)
OUT_BRIEF_TXT.write_text(report_text, encoding="utf-8")

print(report_text)


# ============================================================
# SAVE CSV
# ============================================================

all_runs_df.to_csv(OUT_FULL_CSV, index=False)
brief_summary_df.to_csv(OUT_BRIEF_CSV, index=False)

# Extra CSVs useful for quick terminal inspection
( ROOT / "concept_vs_baseline_global.csv" ).write_text(
    vs_baseline_global_df.to_csv(index=False),
    encoding="utf-8"
)

( ROOT / "concept_by_dataset_scenario.csv" ).write_text(
    by_dataset_scenario_df.to_csv(index=False),
    encoding="utf-8"
)

( ROOT / "concept_leaderboard.csv" ).write_text(
    leaderboard_df.to_csv(index=False),
    encoding="utf-8"
)


# ============================================================
# SAVE XLSX
# ============================================================

def save_xlsx(path: Path, sheets: Dict[str, pd.DataFrame]) -> None:
    try:
        with pd.ExcelWriter(path, engine="openpyxl") as writer:
            for sheet_name, table in sheets.items():
                safe_sheet = sheet_name[:31]
                table.to_excel(writer, sheet_name=safe_sheet, index=False)

        print("Saved XLSX:", path)

    except Exception as e:
        print("Could not save XLSX:", path)
        print("Error:", repr(e))
        print("CSV files are still saved.")
        print("If needed:")
        print("  python -m pip install openpyxl")


full_sheets = {
    "00_overview": overview_df,
    "01_progress": progress_df,
    "02_all_runs": all_runs_df,
    "03_complete_runs": complete_df,
    "04_broken_runs": broken_df,
    "05_epochs_all": epochs_df,
    "06_files_index": files_df,
    "07_leaderboard": leaderboard_df,
    "08_by_scenario_global": by_scenario_global_df,
    "09_by_dataset_scenario": by_dataset_scenario_df,
    "10_by_model_scenario": by_model_scenario_df,
    "11_by_dataset_model": by_dataset_model_scenario_df,
    "12_by_family": by_family_df,
    "13_comparison_vs_baseline": comparison_df,
    "14_vs_baseline_global": vs_baseline_global_df,
    "15_vs_baseline_dataset": vs_baseline_by_dataset_df,
    "16_vs_baseline_ds_model": vs_baseline_by_dataset_model_df,
    "17_best_by_dataset": best_by_dataset_df,
    "18_best_by_model": best_by_model_df,
    "19_best_by_dataset_model": best_by_dataset_model_df,
    "20_best_by_scenario": best_by_scenario_df,
}

brief_sheets = {
    "00_brief_summary": brief_summary_df,
    "01_overview": overview_df,
    "02_top_runs": leaderboard_df.head(100),
    "03_vs_baseline_global": vs_baseline_global_df,
    "04_vs_baseline_dataset": vs_baseline_by_dataset_df,
    "05_best_by_dataset": best_by_dataset_df,
    "06_best_by_dataset_model": best_by_dataset_model_df,
    "07_by_family": by_family_df,
    "08_progress": progress_df,
    "09_broken_runs": broken_df,
}

save_xlsx(OUT_FULL_XLSX, full_sheets)
save_xlsx(OUT_BRIEF_XLSX, brief_sheets)


print("\nDONE")
print("Full CSV :", OUT_FULL_CSV)
print("Full XLSX:", OUT_FULL_XLSX)
print("Brief CSV :", OUT_BRIEF_CSV)
print("Brief XLSX:", OUT_BRIEF_XLSX)
print("Brief TXT :", OUT_BRIEF_TXT)


ROOT: /home/tahiti/Malashin_Projects
RUN_DIR: /home/tahiti/Malashin_Projects/runs_real_text_fractional_concept
Found run dirs: 780
all_runs_df: (780, 81)
epochs_df: (3900, 46)
files_df: (6240, 8)
complete: (780, 81)
broken: (0, 81)
REAL TEXT FRACTIONAL CONCEPT SUMMARY

Run dir: /home/tahiti/Malashin_Projects/runs_real_text_fractional_concept
Expected epochs: 5

Found rows: 780
Complete runs: 780
Non-complete runs: 0
Datasets: 3
Models: 4
Scenarios: 13

BEST SINGLE RUN
----------------------------------------------------------------------------------------------------
ag_news | distilbert_base_uncased | emb_mix_a08_lam015 | run=3 | test_macro_f1@best_val=0.9459 | val_macro_f1=0.9439

TOP SCENARIOS BY MEAN TEST MACRO-F1 @ BEST VAL
----------------------------------------------------------------------------------------------------
emb_replace_a050 | family=embedding_alpha_sweep_replace | mean=0.7867 | std=0.1571 | n=60
emb_replace_a060 | family=embedding_alpha_sweep_replace | mean=0.7852 